# Data Augmentation

This notebook generates synthetic linear voltammogram data in the concentration interval **[7.5, 20] µM** and evaluates whether the increased data volume improves model accuracy.

To do: experimenting with full concentration interval, since after 25 µM there are greater gaps in the data (note: Ip ratio is 1.5x despite 2x concentration jump in 50 to 100µM, linear interpolation will be aproximate, could lead to false records)


### Workflow
1. **Physics-Informed Augmentation**: Interpolation, Gaussian noise, baseline variation, potential drift  
2. **Feature Extraction**: Route synthetic signals through the existing `Signal` + `vectorize` pipeline  
3. **Model Comparison**: Original vs. augmented dataset across all three feature categories  

---
## 1.  Imports & setup

In [1]:
# imports
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Plotly for interactive visualisation
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# filter used for smoothing the voltammogram signals
from scipy.signal import savgol_filter


from sklearn.base import clone
from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import LeaveOneOut, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor

# project specific classes
from voltammogram_signal import Signal
from peak import Peak

In [2]:
# configuration

warnings.filterwarnings('ignore')

# for reproducibility
GLOBAL_RANDOM_SEED = 42
np.random.seed(GLOBAL_RANDOM_SEED)


# default Plotly layout shared by every figure in this notebook
# Inspired by the pplot() helper in voltammogram_signal.py keeps grids visible at minor ticks too
PLOTLY_FIGURE_HEIGHT_PX = 520
PLOTLY_FIGURE_WIDTH_PX = 800
PLOTLY_TEMPLATE = 'plotly_white'

def apply_default_plotly_layout(figure: go.Figure,
                                title_text: str | None = None,
                                xaxis_title: str = 'Potential E (V)',
                                yaxis_title: str = 'Current I (µA)') -> go.Figure:
    """contains figure styling shared by every figure in this notebook"""
    figure.update_layout(
        height=PLOTLY_FIGURE_HEIGHT_PX,
        width=PLOTLY_FIGURE_WIDTH_PX,
        template=PLOTLY_TEMPLATE,
        title=dict(text=title_text, x=0.5, xanchor='center') if title_text else None,
        xaxis_title=xaxis_title,
        yaxis_title=yaxis_title,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1.0),
        margin=dict(l=60, r=30, t=70, b=50),
    )
    figure.update_xaxes(showgrid=True, minor=dict(showgrid=True))
    figure.update_yaxes(showgrid=True, minor=dict(showgrid=True))
    return figure



---

## 2 Load Data


In [3]:
DATASET_PATH = 'datasets/Standard calibration in culture media_extended.xlsx'

raw_calibration_dataframe = pd.read_excel(DATASET_PATH, sheet_name='Raw data')

# row 0 contains units, data starts at row 1
# column 0 = potential axis, column 1 = blank/baseline, columns 2..N = signals.
potential_grid_V = raw_calibration_dataframe.iloc[1:, 0].values.astype(float) # potential array (V)
blank_baseline_current_uA = raw_calibration_dataframe.iloc[1:, 1].values.astype(float) # blank / baseline (µA)
raw_signal_matrix_uA = raw_calibration_dataframe.iloc[1:, 2:].values.astype(float)  # all signal columns (µA)

 # length = 40 (one entry per signal column)
CONCENTRATIONS = [
    100, 100, 100,
    50,  50,  50,
    25,  25,  25,
    15,  15,  15,
    10,  10,  10,
    7.5, 7.5, 7.5,
    5,   5,   5,
    2.5, 2.5,
    1,   1,   1,
    0.5, 0.5, 0.5, 0.5, 0.5, 0.5,
    0.25, 0.25, 0.25,
    0.1,  0.1,  0.1,  0.1,  0.1
] 

print(f"Potential range  : {potential_grid_V.min():.4f} V  ->  {potential_grid_V.max():.4f} V")
print(f"Number of points : {len(potential_grid_V)}")
print(f"Signal columns   : {raw_signal_matrix_uA.shape[1]}")


Potential range  : -0.6001 V  ->  0.5023 V
Number of points : 229
Signal columns   : 40


---
## 3 · Build Anchor Curves at 7.5, 10, 15 µM

We average each triplet of replicates to obtain three high-SNR **anchor curves**
in the augmentation interval [7.5, 20] µM. Anchors are baseline-subtracted so
they represent the Faradaic-only response (the blank is removed once, here,
to avoid re-subtracting it inside the Signal pipeline later).
baseline = background electrical current generated by the culture media itself


**Target interval**: [7.5, 20] µM  
**Base curves**: mean of 3 replicates at 7.5, 10, and 15 µM (after baseline subtraction).

Each synthetic sample is produced by:
1. **Concentration interpolation**: linear mix of the two bounding mean curves, peak current scaled proportionally to concentration (example: a 12µM reading generated using data from 10µM and 15µM by making a new curve) 
2. **Gaussian noise**: white noise with σ = 0.001 µA (strictly limited to preserve chemical validity) 
3. **Baseline variation**: low-order polynomial added to simulate capacitive background distortions (basically see what happens if the base- filler signal varries)
4. **Potential drift**: X-axis shift drawn from *N*(0, σ_drift), σ_drift ~ *U*(0.01, 0.03) (future: can be tested with the dilution dataset)


In [4]:
# build mean base curves for 7.5, 10, 15 µM (avg value for all replicates for each of these concentrations)
def compute_mean_baseline_subtracted_signal(signal_matrix_uA: np.ndarray,
                                            blank_baseline_uA: np.ndarray,
                                            concentration_labels_uM: list[float],
                                            target_concentration_uM: float,
                                            ) -> np.ndarray:
    """For each specific concentration, 
    returns the mean (across replicates) of baseline-subtracted signals. """
    
    # indexes for the target concentration
    column_indices_for_target = [
        i for i, conc in enumerate(concentration_labels_uM)
        if conc == target_concentration_uM
    ]
    
    # subtract baseline to keep only the reaction signal
    baseline_subtracted_replicates = [
        signal_matrix_uA[:, i] - blank_baseline_uA
        for i in column_indices_for_target
    ]
    return np.mean(baseline_subtracted_replicates, axis=0)


# apply method for each of the 3 target concentrations
mean_subtracted_signal_at_7p5uM = compute_mean_baseline_subtracted_signal(
    raw_signal_matrix_uA, blank_baseline_current_uA, CONCENTRATIONS, 7.5
    )
mean_subtracted_signal_at_10uM = compute_mean_baseline_subtracted_signal(
    raw_signal_matrix_uA, blank_baseline_current_uA, CONCENTRATIONS, 10.0
    )
mean_subtracted_signal_at_15uM = compute_mean_baseline_subtracted_signal(
    raw_signal_matrix_uA, blank_baseline_current_uA, CONCENTRATIONS, 15.0
    )


# Sanity check: peak current must grow roughly linearly with concentration
# (for the specific interval)
anchor_curves_summary = []
for concentration_label, anchor_signal in [
    ('7.5 µM', mean_subtracted_signal_at_7p5uM),
    (' 10 µM', mean_subtracted_signal_at_10uM),
    (' 15 µM', mean_subtracted_signal_at_15uM),
]:
    detected_peak = Peak(potential_grid_V, savgol_filter(anchor_signal, 5, 3))
    anchor_curves_summary.append(
        (concentration_label, detected_peak.Ip, detected_peak.Ep)
    )
    print(f'Concentration = {concentration_label}   Ip = {detected_peak.Ip:.4f} µA   Ep = {detected_peak.Ep:.4f} V')


Concentration = 7.5 µM   Ip = 2.7481 µA   Ep = -0.3970 V
Concentration =  10 µM   Ip = 4.0259 µA   Ep = -0.3922 V
Concentration =  15 µM   Ip = 6.1034 µA   Ep = -0.3873 V


### Interactive anchor-curve viewer

A single Plotly figure showing the three anchor curves on a shared axis. Hover
to read exact (E, I) coordinates; zoom-drag to inspect the peak shoulder.

In [5]:
anchor_overview_figure = go.Figure()
anchor_overview_figure.add_trace(go.Scatter(
    x=potential_grid_V, y=mean_subtracted_signal_at_7p5uM,
    mode='lines', name='Anchor 7.5 µM',
    line=dict(color='royalblue', width=2.5),
))
anchor_overview_figure.add_trace(go.Scatter(
    x=potential_grid_V, y=mean_subtracted_signal_at_10uM,
    mode='lines', name='Anchor 10 µM',
    line=dict(color='darkorange', width=2.5),
))
anchor_overview_figure.add_trace(go.Scatter(
    x=potential_grid_V, y=mean_subtracted_signal_at_15uM,
    mode='lines', name='Anchor 15 µM',
    line=dict(color='forestgreen', width=2.5),
))
apply_default_plotly_layout(
    anchor_overview_figure,
    title_text='Anchor curves (baseline-subtracted means of 3 replicates each)',
)
anchor_overview_figure.show()


---
## 4 Augmentation #1 - Concentration Interpolation

**Theoretical basis.** At sub-saturation concentrations of a freely-diffusing
redox species, the Randles-Sevcik equation predicts a peak current
proportional to concentration:

$$ I_p = 2.69 \times 10^{5}\, n^{3/2} A D^{1/2} v^{1/2} C $$

All terms except $C$ are constants for a given experiment, so linearly blending
two anchor curves with weights $(1-\alpha)$ and $\alpha$ should produce a
plausible curve at the intermediate concentration $C_\text{low} + \alpha(C_\text{high} - C_\text{low})$.

This domain-aware interpolation shares its mathematical foundation with data augmentation techniques such as Mixup (mixup: Beyond Empirical Risk Minimization; Hongyi Zhang et al), ensuring that the synthesized signals represent physically valid intermediate states.

Note on extrapolation: 
!* This article explicitly warns that the range for $\lambda \ is [0, 1]$, but for a concentration higher than 15um, lambda exits the valid range. 

The Randles-Sevcik equation guarantees a strict linear relationship (1:1) at low concentrations, as such I implemented a local linear extrapolation (I assume linearity for the entire interval). This linear extrapolation safely assumes the trend holds up to 20 µM. 

However, because this linearity breaks down due to sensor saturation at higher concentrations (the interval 50 µM - 100 µM displays this), a separate approach is required and explored for the full 0.1 µM - 100 µM interval.

**Why this is the foundational step.** 
Noise / baseline-variation / drift only perturb a signal, so they cannot invent a new concentration label. Interpolation is the only way we actually increase label diversity.


In [6]:
def select_bracketing_anchor_curves(target_concentration_uM: float):
    """Return the two anchor curves that bracket `target_concentration_uM`.

    For targets > 15 µM we re-use the 10 -> 15 trend as a *linear extrapolation*
    base (acceptable up to ~20 µM; beyond that the Randles-Sevcik linearity
    starts to fail and a non-linear extrapolation would be needed).
    """
    if target_concentration_uM <= 10.0:
        return 7.5,  mean_subtracted_signal_at_7p5uM, 10.0, mean_subtracted_signal_at_10uM
    elif target_concentration_uM <= 15.0:
        return 10.0, mean_subtracted_signal_at_10uM, 15.0, mean_subtracted_signal_at_15uM
    else:
        # Extrapolation regime - the same anchor pair is used outside [10, 15]
        return 10.0, mean_subtracted_signal_at_10uM, 15.0, mean_subtracted_signal_at_15uM


def linear_interpolate_between_anchors(
        target_concentration_uM: float,
        lower_concentration_uM: float, lower_anchor_signal: np.ndarray,
        upper_concentration_uM: float, upper_anchor_signal: np.ndarray,
        ) -> np.ndarray:
    """Pointwise convex combination of two anchor curves.
    For target=12 µM with lower=10, upper=15: alpha = 0.4
    -> return  0.6 * lower + 0.4 * upper."""
    if upper_concentration_uM == lower_concentration_uM:
         # avoid div by 0, using copy to prevent modifying original arrays
        return lower_anchor_signal.copy()

    # procentile distance of c_target between c_low and c_high
    # For example, if c_target is 12 uM, c_low is 10 uM and c_high is 15 uM,
    #  then alpha = (12 - 10) / (15 - 10) = 0.4
    interpolation_alpha = (
        (target_concentration_uM - lower_concentration_uM)
        / (upper_concentration_uM - lower_concentration_uM)
    )

    # return the weighted average of the two curves
    # knowing alpha = 0.4, we would take 60% of the 10 uM curve and 40% of the 15 uM curve to get the 12 uM curve
    return (1.0 - interpolation_alpha) * lower_anchor_signal + interpolation_alpha * upper_anchor_signal


In [7]:

def interpolate_base_curve(c_target, c_low, I_low, c_high, I_high):
    """
    Linearly interpolate between two base curves so the peak current
    scales proportionally to concentration.
    We calculate the alpha factor (a percentage) that tells us how far c_target is between c_low and c_high,
    For a 12 um curve, how much of the 10um curve and how much of the 15um curve we need to mix to get the curve for the new concentration.
    """
    if c_high == c_low:
        # avoid div by 0, using copy to prevent modifying original arrays
        return I_low.copy()
    # procentile distance of c_target between c_low and c_high
    # For example, if c_target is 12 uM, c_low is 10 uM and c_high is 15 uM, then alpha = (12 - 10) / (15 - 10) = 0.4
    alpha = (c_target - c_low) / (c_high - c_low)
    # return the weighted average of the two curves
    # knowing alpha = 0.4, we would take 60% of the 10 uM curve and 40% of the 15 uM curve to get the 12 uM curve
    return (1.0 - alpha) * I_low + alpha * I_high

# Gaussian Noise 
# uA is strictly limited to preserve chemical validity
NOISE_SIGMA = 0.001

def add_gaussian_noise(I, sigma=NOISE_SIGMA):
    # amount of noise values to add is exactly how many points we have in the signal
    return I + np.random.normal(0.0, sigma, size=I.shape)


### 4.1 Inspection - Interpolation alone (no perturbations)

Three isolated Plotly figures, one diagnostic per chart, so each can be zoomed
without losing the others.


In [8]:
# Generate one synthetic curve at every 0.5 µM step across [7.5, 20]
INTERPOLATION_TARGETS_uM = np.arange(7.5, 20.5, 0.5)
interpolated_currents_per_target = {
    target_concentration_uM: linear_interpolate_between_anchors(
        target_concentration_uM,
        *select_bracketing_anchor_curves(target_concentration_uM)
    )
    for target_concentration_uM in INTERPOLATION_TARGETS_uM
}

# Numeric verification: peak currents must follow the Randles-Sevcik prediction
interpolation_peak_table_rows = []
for target_concentration_uM, interpolated_signal in interpolated_currents_per_target.items():
    detected_peak = Peak(potential_grid_V, savgol_filter(interpolated_signal, 5, 3))
    interpolation_peak_table_rows.append({
        'Target concentration (µM)': target_concentration_uM,
        'Peak current Ip (µA)':      round(detected_peak.Ip, 4),
        'Peak potential Ep (V)':     round(detected_peak.Ep, 4),
        'Region':                    'interpolated' if target_concentration_uM <= 15.0 else 'extrapolated',
    })
interpolation_peak_table = pd.DataFrame(interpolation_peak_table_rows)

# Linearity diagnostic: Pearson r between concentration and peak current
linearity_pearson_r = np.corrcoef(
    interpolation_peak_table['Target concentration (µM)'],
    interpolation_peak_table['Peak current Ip (µA)'],
)[0, 1]
print(f'Pearson r (target vs Ip) over [7.5, 20] µM = {linearity_pearson_r:.6f}')
print(f'(values > 0.9999 confirm Randles-Sevcik linearity of the interpolation)')
interpolation_peak_table.head(10)


Pearson r (target vs Ip) over [7.5, 20] µM = 0.999561
(values > 0.9999 confirm Randles-Sevcik linearity of the interpolation)


,Target concentration (µM),Peak current Ip (µA),Peak potential Ep (V),Region
0,7.5,2.7481,-0.3970,interpolated
1,8.0,2.9913,-0.3922,interpolated
2,8.5,3.2500,-0.3922,interpolated
3,9.0,3.5086,-0.3922,interpolated
4,9.5,3.7673,-0.3922,interpolated
5,10.0,4.0259,-0.3922,interpolated
6,10.5,4.2271,-0.3922,interpolated
7,11.0,4.4328,-0.3873,interpolated
8,11.5,4.6417,-0.3873,interpolated
9,12.0,4.8505,-0.3873,interpolated


#### Figure 4.1.a - All interpolated curves (color = concentration)

In [9]:
interpolation_sweep_figure = go.Figure()

# traces for every interpolated concentration, colour-coded by µM
viridis_palette = [
    f'hsl({240 - 240 * idx / (len(INTERPOLATION_TARGETS_uM) - 1)}, 70%, 45%)'
    for idx in range(len(INTERPOLATION_TARGETS_uM))
]
for target_concentration_uM, color_str in zip(INTERPOLATION_TARGETS_uM, viridis_palette):
    interpolation_sweep_figure.add_trace(go.Scatter(
        x=potential_grid_V,
        y=interpolated_currents_per_target[target_concentration_uM],
        mode='lines',
        line=dict(color=color_str, width=1.2),
        opacity=0.65,
        name=f'{target_concentration_uM:.1f} µM',
        showlegend=False,
        hovertemplate=f'{target_concentration_uM:.1f} µM<br>E=%{{x:.3f}} V<br>I=%{{y:.3f}} µA<extra></extra>',
    ))

# Anchor curves overlaid on top
for anchor_label, anchor_signal, anchor_color in [
    ('Anchor 7.5 µM', mean_subtracted_signal_at_7p5uM, 'royalblue'),
    ('Anchor 10 µM',  mean_subtracted_signal_at_10uM, 'darkorange'),
    ('Anchor 15 µM',  mean_subtracted_signal_at_15uM, 'forestgreen'),
]:
    interpolation_sweep_figure.add_trace(go.Scatter(
        x=potential_grid_V, y=anchor_signal,
        mode='lines', name=anchor_label,
        line=dict(color=anchor_color, width=3.2),
    ))

apply_default_plotly_layout(
    interpolation_sweep_figure,
    title_text='Figure 4.1.a - Interpolated current curves from 7.5 to 20 µM over anchors ',
)
interpolation_sweep_figure.show()


In [10]:
linearity_figure = go.Figure()

# Synthetic Ip values
linearity_figure.add_trace(go.Scatter(
    x=interpolation_peak_table['Target concentration (µM)'],
    y=interpolation_peak_table['Peak current Ip (µA)'],
    mode='markers', name='Interpolated / extrapolated samples',
    marker=dict(color='steelblue', size=8, line=dict(color='black', width=0.6)),
))

# Anchor markers (stars) over the top of the synthetic points
for anchor_label, anchor_curve, anchor_concentration_uM in [
    ('7.5 µM anchor',  mean_subtracted_signal_at_7p5uM,  7.5),
    ('10 µM anchor',   mean_subtracted_signal_at_10uM, 10.0),
    ('15 µM anchor',   mean_subtracted_signal_at_15uM, 15.0),
]:
    anchor_peak_object = Peak(potential_grid_V, savgol_filter(anchor_curve, 5, 3))
    linearity_figure.add_trace(go.Scatter(
        x=[anchor_concentration_uM], y=[anchor_peak_object.Ip],
        mode='markers', name=anchor_label,
        marker=dict(color='crimson', size=18, symbol='star',
                    line=dict(color='black', width=1.2)),
    ))

# Vertical guide at the extrapolation boundary
linearity_figure.add_vline(x=15.0, line=dict(color='gray', dash='dash'),
                            annotation_text='extrapolation boundary',
                            annotation_position='top right')

apply_default_plotly_layout(
    linearity_figure,
    title_text=f'Figure 4.1.b — Ip vs. concentration  (Pearson r = {linearity_pearson_r:.6f})',
    xaxis_title='Target concentration (µM)',
    yaxis_title='Peak current Ip (µA)',
)
linearity_figure.show()


---
## 5 Augmentation #2 - Gaussian Instrument Noise

**Theoretical basis.** 
In reality, no laboratory measuring instrument (such as a picoammeter) records a perfectly smooth curve. There are always unavoidable fluctuations caused by the thermal agitation of the charge carriers inside the physical components (thermal or Johnson noise) and by the analog-to-digital conversion process.

To mimic these real-world instrument imperfections, I artificially add a slight, random "jitter" over the clean measurements, simulating Gaussian noise [1]. 

[1] Wen, Q., Sun, L., Yang, F., Song, X., Gao, J., Wang, X., & Xu, H. (2020). Time Series Data Augmentation for Deep Learning: A Survey.

By adding the noise, I attepmt to prevent the model from simply memorizing the exact training examples (preventing overfitting) and force it to learn the general, robust shape of the pyocyanin peak while ignoring irrelevant variations.


**The problem with the current setting.**
My first iteration used a noise level of 0.001 \mu A$ which is invisible to the naked eye and can render this step useless. (For example, a noise level of $0.001 \mu A$ applied over a typical $4 \mu A$ peak represents a variation of only $0.025\%$.)

Further, I explore the effect of switching the noise level ($\sigma$) across six decades (increasing the amplitude by factors of 10). The goal is to observe, both numerically and visually, the optimal noise threshold that forces the model to generalize better without completely drowning out the underlying analytical signal.


In [11]:
DEFAULT_INSTRUMENT_NOISE_SIGMA_uA = 0.001  # original value

def apply_gaussian_instrument_noise(
    clean_current_signal: np.ndarray,
    noise_sigma_uA: float = DEFAULT_INSTRUMENT_NOISE_SIGMA_uA,
) -> np.ndarray:
    """Add Gaussian noise of standard deviation `noise_sigma_uA` µA to
    every sample of the input signal."""
    additive_noise = np.random.normal(loc=0.0, scale=noise_sigma_uA,
                                       size=clean_current_signal.shape)
    return clean_current_signal + additive_noise


### 5.1 Numerical impact of σ on peak parameters

I pick the 12 µM interpolated curve as the reference, then for each σ in
`{0.001, 0.005, 0.01, 0.05, 0.1, 0.5}` µA I generate **20 noisy realisations**
and measure how σ propagates into Ip, Ep, and FWHM.


In [12]:
NOISE_LEVELS_TO_SWEEP_uA = [0.001, 0.005, 0.01, 0.05, 0.1, 0.5]
NUM_REALISATIONS_PER_NOISE_LEVEL = 20

reference_clean_signal_at_12uM = linear_interpolate_between_anchors(
    12.0, *select_bracketing_anchor_curves(12.0)
)

# --- SETĂRI CRITICE PENTRU CLASA SIGNAL ---
# Trebuie să îi spunem clasei Signal care este grid-ul de potențial global.
Signal.set_common_potential_E(potential_grid_V)
# Foarte important: setăm baseline-ul global ca fiind GOL, pentru ca Signal
# să nu încerce să scadă mediul de cultură din datele noastre deja procesate.
Signal.set_common_baseline_I(np.array([]))

reference_sig_obj = Signal(reference_clean_signal_at_12uM)
reference_Ip = reference_sig_obj.get_peak_current_value()
reference_Ep = reference_sig_obj.get_peak_potential_value()
reference_fwhm = reference_sig_obj.get_peak_fwhm()

print(f'Reference (clean) 12 µM curve:')
print(f'Ip = {reference_Ip:.4f} µA   '
      f'Ep = {reference_Ep:.4f} V   '
      f'FWHM = {reference_fwhm:.4f} V')
print()


# Save & restore RNG so this preview doesn't disturb downstream cells
saved_rng_state = np.random.get_state()
np.random.seed(7)

noise_impact_table_rows = []
for noise_sigma_uA in NOISE_LEVELS_TO_SWEEP_uA:
    noisy_peak_currents      = []
    noisy_peak_potentials    = []
    noisy_peak_fwhms         = []
    for _ in range(NUM_REALISATIONS_PER_NOISE_LEVEL):
        noisy_signal = apply_gaussian_instrument_noise(
            reference_clean_signal_at_12uM, noise_sigma_uA
        )

        noisy_sig_obj = Signal(noisy_signal)

        noisy_peak_currents.append(noisy_sig_obj.get_peak_current_value())
        noisy_peak_potentials.append(noisy_sig_obj.get_peak_potential_value())

        try:
            fwhm_val = noisy_sig_obj.get_peak_fwhm()
        except ValueError:
            fwhm_val = np.nan

        noisy_peak_fwhms.append(fwhm_val)

    noise_impact_table_rows.append({
        'σ (µA)':           noise_sigma_uA,
        'σ / Ip (%)':       round(100 * noise_sigma_uA / reference_Ip, 4),
        'Ip mean (µA)':     round(np.mean(noisy_peak_currents), 4),
        'Ip std (µA)':      round(np.std(noisy_peak_currents),  4),
        '|ΔIp| / Ip (%)':   round(100 * np.mean(np.abs(np.array(noisy_peak_currents)
                                                       - reference_Ip))
                                  / reference_Ip, 4),
        'Ep std (mV)':      round(np.std(noisy_peak_potentials) * 1000, 3),
        'FWHM std (mV)':    round(np.std(noisy_peak_fwhms) * 1000, 3),
    })

np.random.set_state(saved_rng_state)

noise_impact_table = pd.DataFrame(noise_impact_table_rows)
print('Noise level sweep on the 12 µM interpolated curve '
      f'({NUM_REALISATIONS_PER_NOISE_LEVEL} realisations per level):')
display(noise_impact_table)


Reference (clean) 12 µM curve:
Ip = 4.8505 µA   Ep = -0.3873 V   FWHM = 0.0532 V

Noise level sweep on the 12 µM interpolated curve (20 realisations per level):


,σ (µA),σ / Ip (%),Ip mean (µA),Ip std (µA),|ΔIp| / Ip (%),Ep std (mV),FWHM std (mV)
0,0.001,0.0206,4.8505,0.0008,0.0128,0.000,NaN
1,0.005,0.1031,4.8516,0.0037,0.0596,0.000,0.000
2,0.010,0.2062,4.8509,0.0075,0.1299,0.000,NaN
3,0.050,1.0308,4.8563,0.0278,0.4560,2.216,0.000
4,0.100,2.0616,4.8882,0.0655,1.3118,2.306,1.054
5,0.500,10.3082,5.0519,0.3040,6.0465,4.948,4.901


Table interpretation:
σ (µA) [Sigma]: Asta e "butonul de volum" al zgomotului. E parametrul de intrare pe care il dai din cod (0.001, 0.05, 0.1 etc.). Cu cat e mai mare, cu atat semnalul sintetic va "tremura" mai tare pe grafic.

σ / Ip (%): Asta pune zgomotul in perspectiva. Daca varful tau curat are ~4.85 uA inaltime, si tu aplici un zgomot cu o amplitudine de 0.1 uA, coloana asta iti zice: "Acel tremurat de 0.1 reprezinta 2% din inaltimea totala a varfului". E o masuratoare de bun simt ca sa stii cat de intruziv e zgomotul pe care il adaugi.

Ip mean (µA): Tu rulezi testul de 20 de ori cu acelasi zgomot. Asta e media inaltimii peak-ului pe care o gaseste clasa ta Peak in acele 20 de incercari. Ideal, ar trebui sa ramana fix la fel ca varful original (~4.85). Cand zgomotul e prea mare (ex: la 0.5), algoritmul se "impiedica" in tepi de zgomot pe care ii confunda cu varful real, asa ca valoarea trage mult in sus (la 5.05).

Ip std (µA) [Standard Deviation]: In cele 20 de iteratii, inaltimea varfului detectat nu va fi identica mereu, din cauza caracterului random al zgomotului. O data e 4.86, altadata 4.84. Aceasta coloana masoara marja de fluctuatie in inaltime.

|ΔIp| / Ip (%): Aceasta este eroarea absoluta indusa de zgomot. Adica iti spune cu cat la suta e mai departe valoarea medie gasita (cea corupta de zgomot) fata de inaltimea curata, chimic corecta.

Ep std (mV): Pana acum ne-am uitat doar pe axa Y (inaltimea curentului). Asta e eroarea pe axa X (potentialul). Iti arata cat de mult incepe sa se "plimbe" varful la stanga sau la dreapta. O variatie mare aici inseamna ca zgomotul iti muta peak-ul din locul unde ar trebui sa fie fizic.

FWHM std (mV): Fluctutia latimii varfului in acele 20 de teste. Daca vezi NaN (cum a dat in primele teste), inseamna ca forma s-a deformat atat de urat incat metoda clasei tale nu a mai avut puncte valide ca sa extraga o latime logica.

### 5.2 Visualisation - one figure per noise level

To avoid the original overlay problem (where σ = 0.001 µA was pixel-identical to
the clean curve), every σ gets **its own zoomable Plotly figure** with the
clean reference in black and three noisy realisations in colour.


### Inspection - Gaussian Noise

White noise with sigma = 0.001 uA simulates instrument-level current fluctuations. We overlay several stochastic realizations on the 10 uM base curve to confirm the noise is small enough not to corrupt the Faradaic peak. The residual panel (augmented - original) shows the noise alone.


In [13]:
def build_noise_impact_figure(noise_sigma_uA: float, num_realisations: int = 3) -> go.Figure:
    """One isolated figure showing the clean reference and `num_realisations`
    noisy versions at the requested σ."""
    saved_state = np.random.get_state()
    np.random.seed(int(noise_sigma_uA * 1e6) % (2**31))  # deterministic per σ

    figure = go.Figure()

    # Clean reference - thick black
    figure.add_trace(go.Scatter(
        x=potential_grid_V, y=reference_clean_signal_at_12uM,
        mode='lines', name='Clean reference (12 µM)',
        line=dict(color='black', width=2.6),
    ))

    # Coloured realisations
    palette = ['#d62728', '#1f77b4', '#2ca02c', '#ff7f0e', '#9467bd']
    for realisation_index in range(num_realisations):
        noisy_curve = apply_gaussian_instrument_noise(
            reference_clean_signal_at_12uM, noise_sigma_uA
        )
        figure.add_trace(go.Scatter(
            x=potential_grid_V, y=noisy_curve,
            mode='lines', name=f'Realisation #{realisation_index + 1}',
            line=dict(color=palette[realisation_index % len(palette)], width=1.1),
            opacity=0.85,
        ))

    sigma_percent_of_peak = 100 * noise_sigma_uA / reference_Ip
    apply_default_plotly_layout(
        figure,
        title_text=(f'Gaussian noise σ = {noise_sigma_uA} µA   '
                    f'({sigma_percent_of_peak:.2f}% of Ip)'),
    )
    np.random.set_state(saved_state)
    return figure


# Render one separate figure per noise level
for noise_sigma_uA in NOISE_LEVELS_TO_SWEEP_uA:
    build_noise_impact_figure(noise_sigma_uA, num_realisations=3).show()


### 5.3 Residual-only view - the noise itself

For σ at or below ~0.01 µA the noise vanishes against the 4 µA signal. Plotting
the **residual** `ΔI = noisy − clean` on its own axis (in **nA**, not µA) makes the structure unambiguous and confirms it is white-spectrum noise centred on zero.

Plotting the residual on a nA scale simply confirms that the injected noise is purely random and properly centered around zero.

In [14]:
def build_residual_figure(noise_sigma_uA: float, num_realisations: int = 5) -> go.Figure:
    """Plot only the residual (noise itself) on a nA scale."""
    saved_state = np.random.get_state()
    np.random.seed(int(noise_sigma_uA * 1e6) % (2**31))

    figure = go.Figure()
    for realisation_index in range(num_realisations):
        noisy_curve     = apply_gaussian_instrument_noise(
            reference_clean_signal_at_12uM, noise_sigma_uA
        )
        residual_nA     = (noisy_curve - reference_clean_signal_at_12uM) * 1e3  # µA -> nA
        figure.add_trace(go.Scatter(
            x=potential_grid_V, y=residual_nA,
            mode='lines', name=f'Realisation #{realisation_index + 1}',
            line=dict(width=0.9), opacity=0.8,
        ))

    # ±1σ guide lines and zero reference
    one_sigma_nA = noise_sigma_uA * 1e3
    figure.add_hline(y=0.0,           line=dict(color='black', width=1.0))
    figure.add_hline(y= one_sigma_nA, line=dict(color='red', dash='dash', width=1.0),
                     annotation_text=f'+1σ = {one_sigma_nA:.2f} nA',
                     annotation_position='top right')
    figure.add_hline(y=-one_sigma_nA, line=dict(color='red', dash='dash', width=1.0))

    apply_default_plotly_layout(
        figure,
        title_text=f'Residual ΔI at σ = {noise_sigma_uA} µA',
        yaxis_title='Residual ΔI (nA)',
    )
    np.random.set_state(saved_state)
    return figure


for noise_sigma_uA in NOISE_LEVELS_TO_SWEEP_uA:
    build_residual_figure(noise_sigma_uA).show()


In [15]:
scale_reference_figure = go.Figure()
saved_state = np.random.get_state()
np.random.seed(0)
for hypothetical_sigma_uA, trace_color, trace_label in [
    (DEFAULT_INSTRUMENT_NOISE_SIGMA_uA,        'tomato', 'σ = 0.001 µA (base)'),
    (DEFAULT_INSTRUMENT_NOISE_SIGMA_uA * 10,   'blue',   'σ = 0.01 µA  (×10)'),
    (DEFAULT_INSTRUMENT_NOISE_SIGMA_uA * 50,   'green',  'σ = 0.05 µA  (×50)'),
]:
    pure_noise_nA = (np.random.normal(0.0, hypothetical_sigma_uA,
                                       size=potential_grid_V.shape) * 1e3)
    scale_reference_figure.add_trace(go.Scatter(
        x=potential_grid_V, y=pure_noise_nA,
        mode='lines', name=trace_label,
        line=dict(color=trace_color, width=1.0), opacity=0.85,
    ))
np.random.set_state(saved_state)
scale_reference_figure.add_hline(y=0.0, line=dict(color='black', width=1.0))

apply_default_plotly_layout(
    scale_reference_figure,
    title_text=' Pure-noise traces at three σ scales for size reference',
    yaxis_title='Noise ΔI (nA)',
)
scale_reference_figure.show()


---
## 6 · Augmentation #3 - Baseline Variation (polynomial distortion)

**Theoretical basis.** In real-world linear voltammetry, the baseline (non-Faradaic background current) is rarely perfectly flat.  Even for the same concentration, the background current can drift or curve due to surface adsorption or capacitive effects on the electrode. This augmentation step mimics these "non-ideal" baselines to ensure the ML model doesn't expect perfect data.

**The Implementation.**
I simulate the baseline drift by adding a smooth, symetric curve to the original signal. The selected shape of this curve follows the polynomial equation:

$$I_{drift} = E_{norm}^2 - E_{norm}^4$$

**Reasoning for the equation and for normalisation**
Using the normalisation method:
 `E_norm = (E - E.mean()) / (E.max() - E.min())`, 
the normalised window becomes `[−0.5, +0.5]` so the distortion is:

|                       | E_norm = −0.5 | E_norm = 0 | E_norm = +0.5 |
|---|---|---|---|
| E_norm²               | 0.25          | 0          | 0.25          |
| E_norm⁴               | 0.0625        | 0          | 0.0625        |
| **E_norm² − E_norm⁴** | **0.1875**    | **0**      | **0.1875**    |

The polynomial is **symmetric** around the centre and **anchored at zero at the endpoints** (almost — demonstration below). Without the normalisation step (using raw E in [−0.6, +0.5]), the left endpoint would give
0.2304 and the right endpoint 0.1875 → an **asymmetric** distortion.



### Numerical Proof: Why Normalization is Mandatory
Current potential range: -0.6001 V -> 0.5023 V
Exploring an example with and without normalization

Without normalization:
- left (-0.6V): $(-0.6)^2 - (-0.6)^4 = 0.36 - 0.1296 = \mathbf{0.2304}$
- right (+0.5V): $(0.5)^2 - (0.5)^4 = 0.25 - 0.0625 = \mathbf{0.1875}$

As such distortion will be asymetrical without normalization, it would ruin the horisontal aligmnent.

With normalization (using E.max as 0.5V, E.min as -0.6V and E.mean as -0.05V):
- E_norm = (E - E.mean()) / (E.max() - E.min())

- left (-0.6V): $(-0.6 - (-0.05)) / 1.1 = -0.55 / 1.1 = \mathbf{-0.5}$
- right (+0.5V):  $(0.5 - (-0.05)) / 1.1 = 0.55 / 1.1 = \mathbf{+0.5}$
- mid: $(-0.05 - (-0.05)) / 1.1 = \mathbf{0.0}$

As such, the distortion will be simetrical


In [16]:
BASELINE_VARIATION_MAX_AMPLITUDE_uA = 0.05

def apply_polynomial_baseline_distortion(
    clean_current_signal: np.ndarray,
    potential_grid_V: np.ndarray,
    max_amplitude_uA: float = BASELINE_VARIATION_MAX_AMPLITUDE_uA,
) -> np.ndarray:
    """Add a smooth `amp * (E_norm² - E_norm⁴)` polynomial bump to the signal.
    `amp` is drawn from U(-max_amplitude_uA, +max_amplitude_uA) so the distortion
    can be a bump or a dip with equal probability.
    """
    distortion_amplitude_uA = np.random.uniform(-max_amplitude_uA, +max_amplitude_uA)
    # normalisation
    normalised_potential    = (
        (potential_grid_V - potential_grid_V.mean())
        / (potential_grid_V.max() - potential_grid_V.min())
    )
    # distortion
    distortion_curve_uA     = (
        distortion_amplitude_uA
        * (normalised_potential ** 2 - normalised_potential ** 4)
    )
    # return the distorted signal
    return clean_current_signal + distortion_curve_uA

### 6.1 Numerical proof of symmetry and endpoint anchoring

In [17]:
# Compute the shape function (independent of random amplitude) once
normalised_potential_axis = (
    (potential_grid_V - potential_grid_V.mean())
    / (potential_grid_V.max() - potential_grid_V.min())
)
distortion_shape_function = normalised_potential_axis ** 2 - normalised_potential_axis ** 4

# Numerical inspection at key points

# check extreme points
left_endpoint_value     = distortion_shape_function[0]
right_endpoint_value    = distortion_shape_function[-1]

# check centre point (should be close to 0)
centre_index            = np.argmin(np.abs(normalised_potential_axis - 0.0))
centre_value            = distortion_shape_function[centre_index]

# check maximum distortion point (should be around ±0.1)
peak_distortion_index   = np.argmax(np.abs(distortion_shape_function))
peak_distortion_value   = distortion_shape_function[peak_distortion_index]
peak_distortion_E_V     = potential_grid_V[peak_distortion_index]

baseline_shape_report = pd.DataFrame([
    {'Position':  'Left endpoint  (E = -0.6001 V)',
     'E_norm':    round(normalised_potential_axis[0], 4),
     'Shape value (E_norm² - E_norm⁴)': round(left_endpoint_value, 6)},
    {'Position':  'Centre         (E ~ -0.05 V)',
     'E_norm':    round(normalised_potential_axis[centre_index], 4),
     'Shape value (E_norm² - E_norm⁴)': round(centre_value, 6)},
    {'Position':  'Right endpoint (E = +0.5023 V)',
     'E_norm':    round(normalised_potential_axis[-1], 4),
     'Shape value (E_norm² - E_norm⁴)': round(right_endpoint_value, 6)},
    {'Position':  f'Maximum bump   (E = {peak_distortion_E_V:+.4f} V)',
     'E_norm':    round(normalised_potential_axis[peak_distortion_index], 4),
     'Shape value (E_norm² - E_norm⁴)': round(peak_distortion_value, 6)},
])
print('Symmetry & endpoint anchoring check (raw shape function, before amplitude scaling):')
print(f'  Left  endpoint value : {left_endpoint_value:+.6f}')
print(f'  Right endpoint value : {right_endpoint_value:+.6f}')
print(f'  Asymmetry            : {abs(left_endpoint_value - right_endpoint_value):.2e}  '
      f'(target: ~0 i.e. fully symmetric)')
print(f'  Max |distortion|     : {abs(peak_distortion_value):.4f} at E = {peak_distortion_E_V:+.4f} V')
baseline_shape_report


Symmetry & endpoint anchoring check (raw shape function, before amplitude scaling):
  Left  endpoint value : +0.187500
  Right endpoint value : +0.187500
  Asymmetry            : 1.35e-07  (target: ~0 i.e. fully symmetric)
  Max |distortion|     : 0.1875 at E = -0.6001 V


,Position,E_norm,Shape value (E_norm² - E_norm⁴)
0,Left endpoint (E = -0.6001 V),-0.5,0.1875
1,Centre (E ~ -0.05 V),-0.0,0.0000
2,Right endpoint (E = +0.5023 V),0.5,0.1875
3,Maximum bump (E = -0.6001 V),-0.5,0.1875


### Before / after impact on peak parameters

For three amplitude levels (`0.02`, `0.05`, `0.10` µA) we measure how the
distortion propagates into Ip, AUC, and FWHM. This answers the question: *"if the baseline shifts by amp, by how much do the features I care about change?"*

In the Signal class, methods like get_peak_auc() and get_peak_fwhm() are designed for the clean signals. They are window-sensitive, meaning they only look at a very specific range identified by the Peak detection algorithm.

Window Truncation: The polynomial distortion changes the slope of the signal slightly. This causes the Peak detection to "clip" the window too early. In an attempt to use the class methods, they only integrate the "clipped" area, leading up to 80% error.

Local Baseline vs. Global Baseline: The Signal class has a global _normalize_signal method, but it expects a fixed baseline array. In this augmentation attempt, every realization has a randomly different drift. We needed a "Local Zeroing" step (the local_base_offset) to anchor each unique realization back to zero before calculating metrics.

FWHM Search Range: The class method searches for the 50% threshold only within the detected peak window. If the window is too narrow, the "half-maximum" points are technically outside it, causing a ValueError. By searching the entire signal in this script, we ensure the width is always found.

In [21]:
BASELINE_AMPLITUDES_TO_SWEEP_uA = [0.02, 0.05, 0.10]
NUM_REALISATIONS_PER_BASELINE_LEVEL = 30

clean_reference_peak_object_12uM = Signal(savgol_filter(reference_clean_signal_at_12uM, 5, 3))

# Extract metrics to use for error comparison (%)
clean_reference_AUC_12uM = clean_reference_peak_object_12uM.get_peak_auc()
clean_reference_Ip_12uM  = clean_reference_peak_object_12uM.get_peak_current_value()

baseline_impact_table_rows = []
saved_state = np.random.get_state()
np.random.seed(101)

for max_amplitude_uA in BASELINE_AMPLITUDES_TO_SWEEP_uA:
    distorted_peak_currents = []
    distorted_peak_aucs     = []
    distorted_peak_fwhms    = []
    for _ in range(NUM_REALISATIONS_PER_BASELINE_LEVEL):
        # Generate the raw distorted signal (Signal + Polynomial "Bump")
        raw_distorted = apply_polynomial_baseline_distortion(
            reference_clean_signal_at_12uM, potential_grid_V, max_amplitude_uA
        )
        
        # LOCAL ZEROING 
        # WHY = the signal class subtracts a fixed baseline.
        # calculate the average height at the endpoints and subtract it.
        # This re-aligns the "floating" signal back to the zero axis.
        local_base_offset = (raw_distorted[0] + raw_distorted[-1]) / 2
        zeroed_distorted = raw_distorted - local_base_offset
        
        #using the same filter as the reference for consistency
        distorted_sig_obj = Signal(savgol_filter(zeroed_distorted, 5, 3))
        
        # Peak height (Ip) is robust to windowing, so we can get it directly from the Signal object.
        current_Ip = distorted_sig_obj.get_peak_current_value()
        distorted_peak_currents.append(current_Ip)

        # SEPARATE METRICS CALCULATION FROM PEAK DETECTION        

        # Get peak boundaries identified by the object
        start = distorted_sig_obj.peak.start_idx
        end   = distorted_sig_obj.peak.end_idx + 1
        
        # AUC: Integrate the current only within the peak window on the zeroed signal
        # This gives a much more accurate area than the global integration.
        peak_only_auc = np.trapezoid(distorted_sig_obj.I[start:end], Signal.E[start:end])
        distorted_peak_aucs.append(peak_only_auc)

        # FWHM: Search for the 50% threshold across the WHOLE signal array.
        # This prevents 'ValueError' if the peak window is too restrictive.
        threshold = current_Ip * 0.5
        idxs_above = np.where(distorted_sig_obj.I >= threshold)[0]
        
        if len(idxs_above) >= 2:
            # Width = Potential at right-intercept - Potential at left-intercept
            width = Signal.E[idxs_above[-1]] - Signal.E[idxs_above[0]]
            distorted_peak_fwhms.append(width)
        else:
            distorted_peak_fwhms.append(np.nan)
        

    baseline_impact_table_rows.append({
        'max amplitude (µA)':      max_amplitude_uA,
        'amp / Ip (%)':            round(100 * max_amplitude_uA / clean_reference_Ip_12uM, 3),
        'Ip mean (µA)':            round(np.mean(distorted_peak_currents), 4),
        'ΔIp std (µA)':            round(np.std(distorted_peak_currents),  4),
        'AUC mean (µA·V)':         round(np.mean(distorted_peak_aucs), 4),
        'ΔAUC / AUC_clean (%)':    round(
            100 * np.mean(np.abs(np.array(distorted_peak_aucs) - clean_reference_AUC_12uM))
            / clean_reference_AUC_12uM, 3
        ),
        'FWHM std (mV)':           round(np.std(distorted_peak_fwhms) * 1000, 3),
    })

np.random.set_state(saved_state)

baseline_impact_table = pd.DataFrame(baseline_impact_table_rows)
print(f'Clean 12 µM curve:   Ip = {clean_reference_Ip_12uM:.4f} µA   '
      f'AUC = {clean_reference_AUC_12uM:.4f} µA·V')
print(f'\nBaseline-variation impact sweep '
      f'({NUM_REALISATIONS_PER_BASELINE_LEVEL} realisations per level):')
baseline_impact_table


Clean 12 µM curve:   Ip = 4.8475 µA   AUC = 0.3073 µA·V

Baseline-variation impact sweep (30 realisations per level):


,max amplitude (µA),amp / Ip (%),Ip mean (µA),ΔIp std (µA),AUC mean (µA·V),ΔAUC / AUC_clean (%),FWHM std (mV)
0,0.02,0.413,4.8475,0.0011,0.2357,23.452,0.0
1,0.05,1.031,4.8478,0.0031,0.2773,10.443,0.0
2,0.10,2.063,4.8470,0.0055,0.2873,7.245,0.0


In [22]:
# Plot the deterministic shape (positive amplitude only, to read polarity clearly)
distortion_shape_figure = go.Figure()
distortion_shape_figure.add_trace(go.Scatter(
    x=potential_grid_V,
    y=distortion_shape_function,
    mode='lines', name='E_norm² - E_norm⁴',
    line=dict(color='purple', width=2.4),
    fill='tozeroy', fillcolor='rgba(160, 90, 200, 0.18)',
))
distortion_shape_figure.add_hline(y=0.0, line=dict(color='black', width=1.0))
distortion_shape_figure.add_vline(x=potential_grid_V.mean(),
                                   line=dict(color='gray', dash='dot'),
                                   annotation_text='E = E.mean()',
                                   annotation_position='top')

apply_default_plotly_layout(
    distortion_shape_figure,
    title_text='Figure 6.3 — Polynomial shape function (before amplitude scaling)',
    yaxis_title='Distortion shape (unitless)',
)
distortion_shape_figure.show()


### 6.4 Visualisation - one figure per amplitude

For each amplitude we produce **two separate figures** (no overlay, no
side-by-side): (a) the clean curve vs. distorted realisations, and (b) the
extracted distortion ΔI on its own axis.


In [23]:
def build_baseline_variation_overlay_figure(max_amplitude_uA: float,
                                              num_realisations: int = 4) -> go.Figure:
    """
    Creates an Overlay plot to compare the clean reference signal 
    with multiple augmented (distorted) versions.
    """

    saved_state = np.random.get_state()
    np.random.seed(int(max_amplitude_uA * 1e4) % (2**31))

    figure = go.Figure()
    figure.add_trace(go.Scatter(
        x=potential_grid_V, y=reference_clean_signal_at_12uM,
        mode='lines', name='Clean (12 µM)',
        line=dict(color='black', width=2.6),
    ))
    palette = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
    for realisation_index in range(num_realisations):
        # Apply the polynomial distortion
        distorted_curve = apply_polynomial_baseline_distortion(
            reference_clean_signal_at_12uM, potential_grid_V, max_amplitude_uA
        )
        figure.add_trace(go.Scatter(
            x=potential_grid_V, y=distorted_curve,
            mode='lines', name=f'Realisation #{realisation_index + 1}',
            line=dict(color=palette[realisation_index % len(palette)], width=1.4),
            opacity=0.85,
        ))

    apply_default_plotly_layout(
        figure,
        title_text=f'Clean vs. distorted curves   (max amplitude = {max_amplitude_uA} µA)',
    )
    np.random.set_state(saved_state)
    return figure


def build_baseline_variation_residual_figure(max_amplitude_uA: float,
                                              num_realisations: int = 4) -> go.Figure:
    """
    Creates a Residual plot that isolates ONLY the added distortion.
    It shows exactly "what was added" on top of the original signal.
    """
    saved_state = np.random.get_state()
    np.random.seed(int(max_amplitude_uA * 1e4) % (2**31))

    figure = go.Figure()
    palette = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
    for realisation_index in range(num_realisations):
        distorted_curve  = apply_polynomial_baseline_distortion(
            reference_clean_signal_at_12uM, potential_grid_V, max_amplitude_uA
        )
        residual_curve   = distorted_curve - reference_clean_signal_at_12uM
        figure.add_trace(go.Scatter(
            x=potential_grid_V, y=residual_curve,
            mode='lines', name=f'Distortion #{realisation_index + 1}',
            line=dict(color=palette[realisation_index % len(palette)], width=1.4),
            opacity=0.9,
        ))
    figure.add_hline(y=0.0, line=dict(color='black', width=1.0))

    apply_default_plotly_layout(
        figure,
        title_text=f'Extracted distortions only   (max amplitude = {max_amplitude_uA} µA)',
        yaxis_title='Distortion ΔI (µA)',
    )
    np.random.set_state(saved_state)
    return figure


# Render two figures per amplitude (overlay + residual)
for max_amplitude_uA in BASELINE_AMPLITUDES_TO_SWEEP_uA:
    build_baseline_variation_overlay_figure(max_amplitude_uA).show()
    build_baseline_variation_residual_figure(max_amplitude_uA).show()


---
## 7 · Augmentation #4 - Potential (X-axis) Drift

**Theoretical basis.** Two physical effects displace the apparent peak
potential without changing the chemistry:

1. **iR-drop** - voltage lost across the solution resistance, shifting the peak
   to more negative values as current increases.
2. **Reference-electrode drift** - small (<30 mV) random shifts between sweeps.

We model both as a single horizontal translation ΔE drawn from `N(0, σ_drift²)`
with `σ_drift ~ U(0.01, 0.03) V`, then re-sample the shifted curve onto the
**original fixed potential grid** (because the instrument always reports
values at the same E points).


In [24]:
POTENTIAL_DRIFT_SIGMA_LOW_V  = 0.01
POTENTIAL_DRIFT_SIGMA_HIGH_V = 0.03


def apply_horizontal_potential_drift(
    clean_current_signal: np.ndarray,
    potential_grid_V: np.ndarray,
    drift_sigma_low_V:  float = POTENTIAL_DRIFT_SIGMA_LOW_V,
    drift_sigma_high_V: float = POTENTIAL_DRIFT_SIGMA_HIGH_V,
) -> np.ndarray:
    """Shift the signal by ΔE ~ N(0, σ²) along the potential axis, then linearly
    re-interpolate back onto the original grid (endpoints held constant)."""
    drift_sigma_V    = np.random.uniform(drift_sigma_low_V, drift_sigma_high_V)
    delta_potential_V = np.random.normal(loc=0.0, scale=drift_sigma_V)
    shifted_potential_grid_V = potential_grid_V + delta_potential_V
    return np.interp(
        potential_grid_V, shifted_potential_grid_V, clean_current_signal,
        left=clean_current_signal[0], right=clean_current_signal[-1],
    )


### 7.1 Inspection - drift realisations on the 12 µM curve

In [26]:
drift_inspection_figure = go.Figure()

# Clean reference - thick black
drift_inspection_figure.add_trace(go.Scatter(
    x=potential_grid_V, y=reference_clean_signal_at_12uM,
    mode='lines', name='Clean reference (12 µM)',
    line=dict(color='black', width=2.6),
))

# fixed missing ref clean peak
reference_clean_peak = Peak(potential_grid_V, savgol_filter(reference_clean_signal_at_12uM, 5, 3))

# Five drifted realisations, with measured ΔEp annotated in the legend
saved_state = np.random.get_state()
np.random.seed(13)

palette_drift = ['#d62728', '#1f77b4', '#2ca02c', '#ff7f0e', '#9467bd']
recorded_delta_Ep_mV = []
for realisation_index in range(5):
    drifted_signal = apply_horizontal_potential_drift(
        reference_clean_signal_at_12uM, potential_grid_V
    )
    drifted_peak   = Peak(potential_grid_V, savgol_filter(drifted_signal, 5, 3))
    delta_Ep_mV    = (drifted_peak.Ep - reference_clean_peak.Ep) * 1000.0
    recorded_delta_Ep_mV.append(delta_Ep_mV)
    drift_inspection_figure.add_trace(go.Scatter(
        x=potential_grid_V, y=drifted_signal,
        mode='lines',
        name=f'Realisation #{realisation_index + 1}  (ΔEp = {delta_Ep_mV:+.1f} mV)',
        line=dict(color=palette_drift[realisation_index], width=1.3),
        opacity=0.85,
    ))
np.random.set_state(saved_state)

# Vertical guide at the clean Ep
drift_inspection_figure.add_vline(
    x=reference_clean_peak.Ep, line=dict(color='red', dash='dash', width=1.0),
    annotation_text=f'Clean Ep = {reference_clean_peak.Ep:.3f} V',
    annotation_position='top right',
)

apply_default_plotly_layout(
    drift_inspection_figure,
    title_text=f'Figure 7.1 — Potential drift realisations on the 12 µM curve   '
               f'(σ_drift ~ U({POTENTIAL_DRIFT_SIGMA_LOW_V}, {POTENTIAL_DRIFT_SIGMA_HIGH_V}) V)',
)
drift_inspection_figure.show()

print('Measured peak-potential shifts (ΔEp = drifted - clean):')
for realisation_index, delta_Ep_mV in enumerate(recorded_delta_Ep_mV):
    print(f'   Realisation #{realisation_index + 1}:  {delta_Ep_mV:+7.2f} mV')


Measured peak-potential shifts (ΔEp = drifted - clean):
   Realisation #1:   +14.51 mV
   Realisation #2:   -19.34 mV
   Realisation #3:   +62.86 mV
   Realisation #4:   -24.18 mV
   Realisation #5:    -9.67 mV


---
## 9 · Build the Synthetic Dataset

We now generate the **default** augmented dataset (interpolation + all 3
perturbations). The ablation study will later regenerate variants with
individual perturbations enabled or disabled.


In [27]:
NUM_SYNTHETIC_SAMPLES_DEFAULT = 90   # roughly 3 samples per 0.5 µM step across [7.5, 20]


def generate_synthetic_signal_full_pipeline(
    target_concentration_uM: float,
    noise_sigma_uA:    float = DEFAULT_INSTRUMENT_NOISE_SIGMA_uA,
    baseline_amp_uA:   float = BASELINE_VARIATION_MAX_AMPLITUDE_uA,
    enable_drift:      bool  = True,
) -> np.ndarray:
    """Default augmentation pipeline. Each perturbation can be disabled by
    setting its scale parameter to 0 (or, for drift, `enable_drift=False`)."""
    lower_c_uM, lower_anchor, upper_c_uM, upper_anchor = select_bracketing_anchor_curves(
        target_concentration_uM
    )
    synthetic_signal = linear_interpolate_between_anchors(
        target_concentration_uM, lower_c_uM, lower_anchor, upper_c_uM, upper_anchor
    )
    if noise_sigma_uA > 0:
        synthetic_signal = apply_gaussian_instrument_noise(synthetic_signal, noise_sigma_uA)
    if baseline_amp_uA > 0:
        synthetic_signal = apply_polynomial_baseline_distortion(
            synthetic_signal, potential_grid_V, baseline_amp_uA
        )
    if enable_drift:
        synthetic_signal = apply_horizontal_potential_drift(synthetic_signal, potential_grid_V)
    return synthetic_signal


# Re-seed for reproducibility, then sample target concentrations uniformly
np.random.seed(GLOBAL_RANDOM_SEED)
synthetic_target_concentrations_uM = np.sort(
    np.random.uniform(7.5, 20.0, size=NUM_SYNTHETIC_SAMPLES_DEFAULT)
)
synthetic_currents_per_target_uA = [
    generate_synthetic_signal_full_pipeline(target_concentration_uM)
    for target_concentration_uM in synthetic_target_concentrations_uM
]

print(f'Generated {len(synthetic_currents_per_target_uA)} synthetic signals')
print(f'Concentration range: '
      f'{synthetic_target_concentrations_uM.min():.2f} - '
      f'{synthetic_target_concentrations_uM.max():.2f} µM')


Generated 90 synthetic signals
Concentration range: 7.57 - 19.84 µM


### 9.1 Synthetic dataset overview (isolated plots)

In [28]:
# Plot 1: a representative sample of synthetic curves over the anchors
synthetic_overview_figure = go.Figure()
for anchor_label, anchor_signal, anchor_color in [
    ('Anchor 7.5 µM', mean_subtracted_signal_at_7p5uM,  'royalblue'),
    ('Anchor 10 µM',  mean_subtracted_signal_at_10uM, 'darkorange'),
    ('Anchor 15 µM',  mean_subtracted_signal_at_15uM, 'forestgreen'),
]:
    synthetic_overview_figure.add_trace(go.Scatter(
        x=potential_grid_V, y=anchor_signal, mode='lines',
        name=anchor_label, line=dict(color=anchor_color, width=2.8),
    ))

sampled_indices = np.linspace(0, NUM_SYNTHETIC_SAMPLES_DEFAULT - 1, 6, dtype=int)
for sample_position, sample_index in enumerate(sampled_indices):
    sample_target_uM = synthetic_target_concentrations_uM[sample_index]
    synthetic_overview_figure.add_trace(go.Scatter(
        x=potential_grid_V, y=synthetic_currents_per_target_uA[sample_index],
        mode='lines', name=f'Synth {sample_target_uM:.2f} µM',
        line=dict(width=1.1, dash='dot'),
        opacity=0.75,
    ))

apply_default_plotly_layout(
    synthetic_overview_figure,
    title_text='Synthetic curves over anchors',
)
synthetic_overview_figure.show()


# Plot 2: linearity check on the synthetic dataset
synthetic_peak_currents_uA = [
    Peak(potential_grid_V, savgol_filter(synthetic_signal, 5, 3)).Ip
    for synthetic_signal in synthetic_currents_per_target_uA
]
synthetic_linearity_figure = go.Figure()
synthetic_linearity_figure.add_trace(go.Scatter(
    x=synthetic_target_concentrations_uM, y=synthetic_peak_currents_uA,
    mode='markers', name='Synthetic',
    marker=dict(color='steelblue', size=6, opacity=0.7),
))
for anchor_curve, anchor_concentration_uM, anchor_label in [
    (mean_subtracted_signal_at_7p5uM,  7.5,  '7.5 µM anchor'),
    (mean_subtracted_signal_at_10uM, 10.0,  '10 µM anchor'),
    (mean_subtracted_signal_at_15uM, 15.0,  '15 µM anchor'),
]:
    anchor_peak_obj = Peak(potential_grid_V, savgol_filter(anchor_curve, 5, 3))
    synthetic_linearity_figure.add_trace(go.Scatter(
        x=[anchor_concentration_uM], y=[anchor_peak_obj.Ip],
        mode='markers', name=anchor_label,
        marker=dict(color='crimson', size=16, symbol='star',
                    line=dict(color='black', width=1.0)),
    ))
synthetic_linearity_figure.add_vline(
    x=15.0, line=dict(color='gray', dash='dash'),
    annotation_text='extrapolation boundary', annotation_position='top right',
)
apply_default_plotly_layout(
    synthetic_linearity_figure,
    title_text='Linearity of Ip vs. concentration (synthetic dataset)',
    xaxis_title='Concentration (µM)',
    yaxis_title='Peak current Ip (µA)',
)
synthetic_linearity_figure.show()


---
## 10 · Feature Extraction Pipeline

Synthetic signals are routed through the same Signal + vectorize pipeline used in feature_extraction.ipynb.

 **Note**: Augmented signals are already baseline-subtracted. We temporarily clear `Signal.I_baseline` before processing them, then restore it.


In [29]:
def vectorize_signal_features(signal_object: 'Signal', feature_suite: str = 'core') -> list:
    """Extract a flat feature vector from a Signal object for one of the
    three named suites ('core', 'extended', 'experimental')."""
    feature_vector = []
    if feature_suite in ('core', 'extended', 'experimental'):
        feature_vector += [
            signal_object.get_peak_current_value(),
            signal_object.get_peak_potential_value(),
            signal_object.get_peak_auc(),
            signal_object.get_peak_fwhm(),
        ]
    if feature_suite in ('extended', 'experimental'):
        feature_vector += [
            signal_object.get_pca1_comp(),
            signal_object.get_first_derivative_max(),
            signal_object.get_second_derivative_min(),
        ]
    if feature_suite == 'experimental':
        feature_vector += [
            signal_object.get_left_slope(),
            signal_object.get_right_slope(),
            signal_object.get_asymetry(),
            signal_object.get_peak_sharpness(),
            signal_object.get_peak_compactness(),
            signal_object.get_current_variance(),
            signal_object.get_peak_skewness(),
            signal_object.get_peak_kurtosis(),
            signal_object.get_tchebichef_curve_moments(),
            signal_object.get_mean_peak(),
            signal_object.get_signal_entropy(),
            signal_object.get_spectral_entropy(),
            signal_object.get_fft_power(),
            signal_object.get_pca2_comp(),
            signal_object.get_pca3_comp(),
            signal_object.get_wavelet_energy(),
        ]
    return feature_vector


FEATURE_COLUMN_NAMES_BY_SUITE = {
    'core': ['peak_current', 'peak_potential', 'peak_AUC', 'peak_FWHM'],
    'extended': ['peak_current', 'peak_potential', 'peak_AUC', 'peak_FWHM',
                 'pca1_comp', 'first_derivative_max', 'second_derivative_min'],
    'experimental': ['peak_current', 'peak_potential', 'peak_AUC', 'peak_FWHM',
                     'pca1_comp', 'first_derivative_max', 'second_derivative_min',
                     'left_slope', 'right_slope', 'asymetry', 'peak_sharpness',
                     'peak_compactness', 'current_variance', 'peak_skewness',
                     'peak_kurtosis', 'tchebichef_curve_moments', 'mean_peak',
                     'signal_entropy', 'spectral_entropy', 'fft_power',
                     'pca2_comp', 'pca3_comp', 'wavelet_energy'],
}


In [30]:
# Process ORIGINAL signals (baseline subtraction handled by Signal)
Signal.set_common_potential_E(potential_grid_V)
Signal.set_common_baseline_I(blank_baseline_current_uA)
Signal.sig_id = 1

original_signals_with_labels = []
for column_index in range(raw_signal_matrix_uA.shape[1]):
    try:
        sig = Signal(raw_signal_matrix_uA[:, column_index])
        original_signals_with_labels.append((sig, CONCENTRATIONS[column_index]))
    except Exception as e:
        print(f"  [WARN] signal {column_index} skipped: {e}")

print(f"Original signals processed: {len(original_signals_with_labels)}")


Original signals processed: 40


In [31]:
# Process AUGMENTED signals
# Augmented signals are already baseline-subtracted; disable auto-subtraction.

def process_synthetic_currents_into_signal_objects(
        synthetic_current_signals: list[np.ndarray],
        synthetic_target_concentrations_uM: np.ndarray,
        ) -> tuple[list[tuple['Signal', float]], int]:
    
    Signal.set_common_baseline_I(np.array([]))
    signal_objects_with_labels: list[tuple['Signal', float]] = []
    
    num_skipped = 0
    for synthetic_current_signal, target_concentration_uM in zip(
        synthetic_current_signals, synthetic_target_concentrations_uM
    ):
        try:
            signal_objects_with_labels.append(
                (Signal(synthetic_current_signal), float(target_concentration_uM))
            )
        except Exception:
            num_skipped += 1
    Signal.set_common_baseline_I(blank_baseline_current_uA) # restore
    return signal_objects_with_labels, num_skipped

default_synthetic_signal_objects, default_num_skipped = (
    process_synthetic_currents_into_signal_objects(
        synthetic_currents_per_target_uA, synthetic_target_concentrations_uM
    )
)
print(f'Default synthetic signals processed: '
      f'{len(default_synthetic_signal_objects)}  (skipped: {default_num_skipped})')


Default synthetic signals processed: 90  (skipped: 0)


In [32]:
# DataFrames for all three feature suites
def build_feature_dataframe(
        signal_objects_with_labels: list[tuple['Signal', float]],
        feature_suite: str,
    ) -> pd.DataFrame:
    """Vectorise every signal and assemble a DataFrame whose last column is the
    concentration target."""
    feature_rows = []
    num_skipped = 0
    for signal_obj, target_concentration_uM in signal_objects_with_labels:
        try:
            feature_rows.append(
                vectorize_signal_features(signal_obj, feature_suite)
                + [target_concentration_uM])
        except Exception as e:
            num_skipped += 1
    
    if num_skipped > 0:
        print(f">>> [{feature_suite}] Skipped {num_skipped} signals due to extraction errors.")
    return pd.DataFrame(
        feature_rows, 
        columns=FEATURE_COLUMN_NAMES_BY_SUITE[feature_suite] + ['concentration']
    )


original_features_core_df         = build_feature_dataframe(original_signals_with_labels, 'core')
original_features_extended_df     = build_feature_dataframe(original_signals_with_labels, 'extended')
original_features_experimental_df = build_feature_dataframe(original_signals_with_labels, 'experimental')

synthetic_features_core_df_default         = build_feature_dataframe(default_synthetic_signal_objects, 'core')
synthetic_features_extended_df_default     = build_feature_dataframe(default_synthetic_signal_objects, 'extended')
synthetic_features_experimental_df_default = build_feature_dataframe(default_synthetic_signal_objects, 'experimental')

combined_core = pd.concat([original_features_core_df, synthetic_features_core_df_default], ignore_index=True)
combined_ext  = pd.concat([original_features_extended_df, synthetic_features_extended_df_default], ignore_index=True)
combined_exp  = pd.concat([original_features_experimental_df, synthetic_features_experimental_df_default], ignore_index=True)

print(f'Original   - core: {original_features_core_df.shape}  '
      f'| extended: {original_features_extended_df.shape}  '
      f'| experimental: {original_features_experimental_df.shape}')
print(f'Synthetic  - core: {synthetic_features_core_df_default.shape}  '
      f'| extended: {synthetic_features_extended_df_default.shape}  '
      f'| experimental: {synthetic_features_experimental_df_default.shape}')

>>> [core] Skipped 27 signals due to extraction errors.
>>> [extended] Skipped 27 signals due to extraction errors.
>>> [experimental] Skipped 27 signals due to extraction errors.
Original   - core: (40, 5)  | extended: (40, 8)  | experimental: (40, 24)
Synthetic  - core: (63, 5)  | extended: (63, 8)  | experimental: (63, 24)


---
## 11 · Ablation Study - Which Augmentation Actually Helps?

This section answers the question: **which of
the four augmentation methods improves accuracy, by how much, and which one silently hurts**.

### Design

For each *strategy* listed below we regenerate **a fresh synthetic dataset
from scratch** (same target concentrations, same number of samples, same
RNG seed) so the only difference between strategies is the augmentation
recipe. Each dataset is then evaluated under **Protocol C** (held-out
original sample → trained on remaining originals + the full synthetic set).
Protocol C is the only protocol that gives an unbiased estimate of
augmentation benefit because synthetic samples never enter the test set.

| Strategy ID | Interpolation | Noise σ (µA) | Baseline amp (µA) | Potential drift |
|---|---|---|---|---|
| **S0_baseline_no_aug**     | -    | -     | -    | -   |
| **S1_interp_only**         | ✔    | 0     | 0    | ✘   |
| **S2_interp_noise_0p001**  | ✔    | 0.001 | 0    | ✘   |
| **S3_interp_noise_0p005**  | ✔    | 0.005 | 0    | ✘   |
| **S4_interp_noise_0p01**   | ✔    | 0.01  | 0    | ✘   |
| **S5_interp_noise_0p05**   | ✔    | 0.05  | 0    | ✘   |
| **S6_interp_noise_0p1**    | ✔    | 0.1   | 0    | ✘   |
| **S7_interp_baseline_only**| ✔    | 0     | 0.05 | ✘   |
| **S8_interp_drift_only**   | ✔    | 0     | 0    | ✔   |
| **S9_all_default**         | ✔    | 0.001 | 0.05 | ✔   |


### Evaluation Protocols

| Protocol | Description |
|---|---|
| **A** | LOOCV on original data only (40 samples) - baseline |
| **B** | LOOCV on combined dataset (original + synthetic) |
| **C** | Leave out one original sample; train on remaining originals + ALL synthetic (conservative, unbiased) |

**Protocol C** is the scientifically cleanest comparison since synthetic samples never appear in the test set.

**Protocol B** runs high risk of data leakage, counter-example of data augumentation (the test signal could be an augumented 10uM with some noise, but in training the model receives the source signal from which the augumented one was derived)


In [33]:
# Define the ablation strategies as a structured list
ABLATION_STRATEGIES_CONFIG = [
    # (strategy_id, use_interpolation, noise_sigma_uA, baseline_amp_uA, enable_drift)
    ('S0_baseline_no_aug',     False, 0.0,   0.0,  False),
    ('S1_interp_only',         True,  0.0,   0.0,  False),
    ('S2_interp_noise_0p001',  True,  0.001, 0.0,  False),
    ('S3_interp_noise_0p005',  True,  0.005, 0.0,  False),
    ('S4_interp_noise_0p01',   True,  0.01,  0.0,  False),
    ('S5_interp_noise_0p05',   True,  0.05,  0.0,  False),
    ('S6_interp_noise_0p1',    True,  0.1,   0.0,  False),
    ('S7_interp_baseline_only',True,  0.0,   0.05, False),
    ('S8_interp_drift_only',   True,  0.0,   0.0,  True),
    ('S9_all_default',         True,  0.001, 0.05, True),
]

def generate_synthetic_dataset_for_strategy(
        use_interpolation: bool,
        noise_sigma_uA:    float,
        baseline_amp_uA:   float,
        enable_drift:      bool,
        target_concentrations_uM: np.ndarray,
        rng_seed: int = GLOBAL_RANDOM_SEED,
    ) -> list[np.ndarray]:
    """Return a list of current-signal arrays generated under the strategy.
    `use_interpolation=False` returns an empty list (no synthetic data)."""
    if not use_interpolation:
        return []
    np.random.seed(rng_seed)
    generated_signals = []
    for target_concentration_uM in target_concentrations_uM:
        lower_c, lower_a, upper_c, upper_a = select_bracketing_anchor_curves(
            target_concentration_uM
        )
        synthetic_signal = linear_interpolate_between_anchors(
            target_concentration_uM, lower_c, lower_a, upper_c, upper_a
        )
        if noise_sigma_uA > 0:
            synthetic_signal = apply_gaussian_instrument_noise(
                synthetic_signal, noise_sigma_uA
            )
        if baseline_amp_uA > 0:
            synthetic_signal = apply_polynomial_baseline_distortion(
                synthetic_signal, potential_grid_V, baseline_amp_uA
            )
        if enable_drift:
            synthetic_signal = apply_horizontal_potential_drift(
                synthetic_signal, potential_grid_V
            )
        generated_signals.append(synthetic_signal)
    return generated_signals


In [34]:
# Evaluation helpers (Protocol A and Protocol C only - B is the risky one)
def evaluate_protocol_A_loocv_original_only(
    feature_dataframe: pd.DataFrame,
    feature_column_names: list[str],
    model_to_clone,
) -> dict:
    """LOOCV on the original-only DataFrame."""
    X_features          = feature_dataframe[feature_column_names]
    y_concentrations_uM = feature_dataframe['concentration']
    true_values, predicted_values = [], []
    for train_indices, test_indices in LeaveOneOut().split(X_features):
        model_instance = clone(model_to_clone)
        model_instance.fit(X_features.iloc[train_indices], y_concentrations_uM.iloc[train_indices])
        predicted_values.append(model_instance.predict(X_features.iloc[test_indices])[0])
        true_values.append(y_concentrations_uM.iloc[test_indices].iloc[0])
    true_array, pred_array = np.array(true_values), np.array(predicted_values)
    return dict(
        MAE=float(mean_absolute_error(true_array, pred_array)),
        RMSE=float(np.sqrt(mean_squared_error(true_array, pred_array))),
        R2=float(r2_score(true_array, pred_array)),
    )


def evaluate_protocol_C_held_out_original_plus_synthetic(
    original_feature_dataframe:   pd.DataFrame,
    synthetic_feature_dataframe:  pd.DataFrame,
    feature_column_names:         list[str],
    model_to_clone,
) -> dict:
    """Hold out one original sample at a time; train on the remaining originals
    + the entire synthetic dataset; test on the held-out original."""
    X_original  = original_feature_dataframe[feature_column_names]
    y_original  = original_feature_dataframe['concentration']
    X_synthetic = synthetic_feature_dataframe[feature_column_names]
    y_synthetic = synthetic_feature_dataframe['concentration']

    true_values, predicted_values = [], []
    for held_out_index in range(len(X_original)):
        kept_original_indices = [i for i in range(len(X_original)) if i != held_out_index]
        X_training = pd.concat(
            [X_original.iloc[kept_original_indices], X_synthetic], ignore_index=True,
        )
        y_training = pd.concat(
            [y_original.iloc[kept_original_indices], y_synthetic], ignore_index=True,
        )
        model_instance = clone(model_to_clone)
        model_instance.fit(X_training, y_training)
        predicted_values.append(model_instance.predict(X_original.iloc[[held_out_index]])[0])
        true_values.append(y_original.iloc[held_out_index])
    true_array, pred_array = np.array(true_values), np.array(predicted_values)
    return dict(
        MAE=float(mean_absolute_error(true_array, pred_array)),
        RMSE=float(np.sqrt(mean_squared_error(true_array, pred_array))),
        R2=float(r2_score(true_array, pred_array)),
    )


In [35]:
# model configurations for the ablation (same across all feature suites and protocols for consistency)
ABLATION_MODELS = {
    'Ridge': Ridge(alpha=0.1, fit_intercept=False, random_state=42),
    'DecisionTree': DecisionTreeRegressor(
        max_depth=None, min_samples_split=2, min_samples_leaf=1, random_state=42
    ),
    'RandomForest': RandomForestRegressor(
        n_estimators=500, max_depth=9, max_features=0.75,
        min_samples_split=2, min_samples_leaf=1, random_state=42, n_jobs=-1,
    ),
    'XGBoost': XGBRegressor(
        n_estimators=235, max_depth=3, learning_rate=0.0698,
        subsample=0.7682, colsample_bytree=0.9395,
        gamma=0.3391, min_child_weight=3,
        random_state=42, verbosity=0, n_jobs=-1,
    ),
}

# for starters, i used the experimental feature suite by default for the ablation - it has the
# largest dynamic range and is the most sensitive to augmentation choices.
# ABLATION_FEATURE_SUITE = 'experimental'
# ABLATION_FEATURE_COLUMN_NAMES = FEATURE_COLUMN_NAMES_BY_SUITE[ABLATION_FEATURE_SUITE]
FEATURE_SUITES_TO_EVALUATE = ['core', 'extended', 'experimental']

In [ ]:
# RUNNING THE ABLATION STUDY
ablation_result_rows = []

for feature_suite in FEATURE_SUITES_TO_EVALUATE:
    print('=' * 90)
    print(f' STARTING ABLATION STUDY FOR FEATURE SUITE: {feature_suite.upper()}')
    print('=' * 90)
    if feature_suite == 'core':
        current_original_df = original_features_core_df
    elif feature_suite == 'extended':
        current_original_df = original_features_extended_df
    else:
        current_original_df = original_features_experimental_df

    current_feature_column_names = FEATURE_COLUMN_NAMES_BY_SUITE[feature_suite]
    
    # Pre-compute Protocol A (it does not depend on the synthetic data and we want
    # it as the absolute baseline that every strategy is compared against)
    print('Computing Protocol A baseline (original-only LOOCV)...')
    protocol_A_by_model = {}
    for model_name, model_object in ABLATION_MODELS.items():
        protocol_A_by_model[model_name] = evaluate_protocol_A_loocv_original_only(
            current_original_df,
            current_feature_column_names,
            model_object,
        )
        print(f'   {model_name:14s}  '
            f'MAE={protocol_A_by_model[model_name]["MAE"]:.4f}   '
            f'RMSE={protocol_A_by_model[model_name]["RMSE"]:.4f}   '
            f'R2={protocol_A_by_model[model_name]["R2"]:.4f}')
    print()

    # Iterate strategies
    for (strategy_id,
        use_interpolation,
        noise_sigma_uA,
        baseline_amp_uA,
        enable_drift) in ABLATION_STRATEGIES_CONFIG:

        print(f'--- Strategy {strategy_id} '
            f'(noise={noise_sigma_uA}, baseline_amp={baseline_amp_uA}, drift={enable_drift}) ---')


        # Generate synthetic dataset (or skip for S0 = no augmentation)
        synthetic_current_signals = generate_synthetic_dataset_for_strategy(
            use_interpolation=use_interpolation,
            noise_sigma_uA=noise_sigma_uA,
            baseline_amp_uA=baseline_amp_uA,
            enable_drift=enable_drift,
            target_concentrations_uM=synthetic_target_concentrations_uM,
            rng_seed=GLOBAL_RANDOM_SEED,
        )

        if not use_interpolation:
            # S0: report Protocol A as both baseline and "augmented" performance since the data is identic
            for model_name in ABLATION_MODELS:
                metrics_A = protocol_A_by_model[model_name]
                ablation_result_rows.append({
                    'Feature Suite':      feature_suite,
                    'Strategy':           strategy_id,
                    'Model':              model_name,
                    'Noise σ (µA)':       noise_sigma_uA,
                    'Baseline amp (µA)':  baseline_amp_uA,
                    'Drift':              enable_drift,
                    'Protocol A MAE':     round(metrics_A['MAE'],  4),
                    'Protocol A R2':      round(metrics_A['R2'],   4),
                    'Protocol C MAE':     round(metrics_A['MAE'],  4),  # no synthetic data
                    'Protocol C RMSE':    round(metrics_A['RMSE'], 4),
                    'Protocol C R2':      round(metrics_A['R2'],   4),
                    'ΔMAE (C - A)':       0.0,
                    'ΔR2  (C - A)':       0.0,
                })
            continue

        # Route synthetic currents through Signal pipeline
        synthetic_signal_objects, _ = process_synthetic_currents_into_signal_objects(
            synthetic_current_signals, synthetic_target_concentrations_uM,
        )
        synthetic_feature_dataframe = build_feature_dataframe(
            synthetic_signal_objects, feature_suite,
        )

        # evaluate Protocol C for every model
        for model_name, model_object in ABLATION_MODELS.items():
            metrics_A = protocol_A_by_model[model_name]
            metrics_C = evaluate_protocol_C_held_out_original_plus_synthetic(
                original_feature_dataframe=current_original_df,
                synthetic_feature_dataframe=synthetic_feature_dataframe,
                feature_column_names=current_feature_column_names,
                model_to_clone=model_object,
            )
            ablation_result_rows.append({
                'Feature Suite':      feature_suite,
                'Strategy':           strategy_id,
                'Model':              model_name,
                'Noise σ (µA)':       noise_sigma_uA,
                'Baseline amp (µA)':  baseline_amp_uA,
                'Drift':              enable_drift,
                'Protocol A MAE':     round(metrics_A['MAE'],  4),
                'Protocol A R2':      round(metrics_A['R2'],   4),
                'Protocol C MAE':     round(metrics_C['MAE'],  4),
                'Protocol C RMSE':    round(metrics_C['RMSE'], 4),
                'Protocol C R2':      round(metrics_C['R2'],   4),
                'ΔMAE (C - A)':       round(metrics_C['MAE'] - metrics_A['MAE'], 4),
                'ΔR2  (C - A)':       round(metrics_C['R2']  - metrics_A['R2'],  4),
            })
        print()

ablation_results_dataframe = pd.DataFrame(ablation_result_rows)
ablation_results_dataframe.to_csv('augmentation_ablation_results.csv', index=False)
print('Ablation complete - results saved to augmentation_ablation_results_all_suites.csv')


 STARTING ABLATION STUDY FOR FEATURE SUITE: CORE
Computing Protocol A baseline (original-only LOOCV)...
   Ridge           MAE=1.0618   RMSE=1.8811   R2=0.9953
   DecisionTree    MAE=0.0625   RMSE=0.3953   R2=0.9998
   RandomForest    MAE=0.9875   RMSE=2.2354   R2=0.9934
   XGBoost         MAE=2.9404   RMSE=6.8252   R2=0.9380

--- Strategy S0_baseline_no_aug (noise=0.0, baseline_amp=0.0, drift=False) ---
--- Strategy S1_interp_only (noise=0.0, baseline_amp=0.0, drift=False) ---
>>> [core] Skipped 5 signals due to extraction errors.

--- Strategy S2_interp_noise_0p001 (noise=0.001, baseline_amp=0.0, drift=False) ---
>>> [core] Skipped 1 signals due to extraction errors.

--- Strategy S3_interp_noise_0p005 (noise=0.005, baseline_amp=0.0, drift=False) ---
>>> [core] Skipped 1 signals due to extraction errors.

--- Strategy S4_interp_noise_0p01 (noise=0.01, baseline_amp=0.0, drift=False) ---

--- Strategy S5_interp_noise_0p05 (noise=0.05, baseline_amp=0.0, drift=False) ---

--- Strategy S6

In [41]:
# Find the best and worst strategy per model based on Protocol C MAE (only experimental features suite)
print('Per-model leaderboard by Feature Suite:')
for feature_suite in FEATURE_SUITES_TO_EVALUATE:
    print('=' * 90)
    print(f' FEATURE SUITE: {feature_suite.upper()}')
    print('=' * 90)
    suite_df = ablation_results_dataframe[ablation_results_dataframe['Feature Suite'] == feature_suite]
    
    for model_name in ABLATION_MODELS.keys():
        rows_for_this_model = (
            suite_df[suite_df['Model'] == model_name]
            .sort_values('Protocol C MAE')
            .reset_index(drop=True)
        )
        best_row   = rows_for_this_model.iloc[0]
        worst_row  = rows_for_this_model.iloc[-1]
        baseline_A = rows_for_this_model[
            rows_for_this_model['Strategy'] == 'S0_baseline_no_aug'
        ].iloc[0]
        all_default = rows_for_this_model[
            rows_for_this_model['Strategy'] == 'S9_all_default'
        ].iloc[0]

        print(f'\n{model_name}:')
        print(f'   Baseline (no aug)    : MAE = {baseline_A["Protocol C MAE"]:.4f}   R2 = {baseline_A["Protocol C R2"]:.4f}')
        print(f'   All-default pipeline : MAE = {all_default["Protocol C MAE"]:.4f}   R2 = {all_default["Protocol C R2"]:.4f}   '
            f'(Δ = {all_default["ΔMAE (C - A)"]:+.4f})')
        print(f'   --> BEST strategy    : {best_row["Strategy"]:25s}  '
            f'MAE = {best_row["Protocol C MAE"]:.4f}   ΔMAE = {best_row["ΔMAE (C - A)"]:+.4f}')
        print(f'   --> WORST strategy   : {worst_row["Strategy"]:25s}  '
            f'MAE = {worst_row["Protocol C MAE"]:.4f}   ΔMAE = {worst_row["ΔMAE (C - A)"]:+.4f}')
    print('=' * 90)


Per-model leaderboard by Feature Suite:
 FEATURE SUITE: CORE

Ridge:
   Baseline (no aug)    : MAE = 1.0618   R2 = 0.9953
   All-default pipeline : MAE = 1.2084   R2 = 0.9941   (Δ = +0.1466)
   --> BEST strategy    : S0_baseline_no_aug         MAE = 1.0618   ΔMAE = +0.0000
   --> WORST strategy   : S6_interp_noise_0p1        MAE = 1.2360   ΔMAE = +0.1742

DecisionTree:
   Baseline (no aug)    : MAE = 0.0625   R2 = 0.9998
   All-default pipeline : MAE = 0.0111   R2 = 1.0000   (Δ = -0.0514)
   --> BEST strategy    : S9_all_default             MAE = 0.0111   ΔMAE = -0.0514
   --> WORST strategy   : S3_interp_noise_0p005      MAE = 0.6536   ΔMAE = +0.5911

RandomForest:
   Baseline (no aug)    : MAE = 0.9875   R2 = 0.9934
   All-default pipeline : MAE = 0.8261   R2 = 0.9935   (Δ = -0.1614)
   --> BEST strategy    : S9_all_default             MAE = 0.8261   ΔMAE = -0.1614
   --> WORST strategy   : S0_baseline_no_aug         MAE = 0.9875   ΔMAE = +0.0000

XGBoost:
   Baseline (no aug)    : M

### For suite core:
#### Where data augmentation thrives:
- XGBoost
The default / mixed data pipeline (S9_all_default) dropped the MAE from 2.9404 to 2.5439 (Improvement: -0.3965).

- RandomForest
The default / mixed data pipeline (S9_all_default) dropped the MAE from 0.9875 to 0.8261 (Improvement: -0.1614).

#### Where data augmentation hurt:

- DecisionTree
The intermediate noise strategy (S3_interp_noise_0p005) caused the model to overfit to random jitters rather than the underlying trend. Raised the MAE from 0.0625 to 0.6536 (Degradation: +0.5911).


- Ridge
The maximum noise strategy (S6_interp_noise_0p1) injected too much variance. raised the MAE from 1.0618 to 1.2360 (Degradation: +0.1742).


### For suite extended:
#### Where data augmentation thrives:
- XGBoost
The default / mixed data pipeline (S9_all_default) dropped the MAE from 2.0778 to 1.7837 (Improvement: -0.2941).

- Ridge
The default / mixed data pipeline (S9_all_default) dropped the MAE from 1.4705 to 1.2821 (Improvement: -0.1884).

#### Where data augmentation hurt:

- DecisionTree
The maximum noise strategy (S6_interp_noise_0p1) completely derailed the logic with complex derivative features. raised the MAE from 0.0100 to 1.4141 (Degradation: +1.4041)

- RandomForest
The slight noise strategy (S3_interp_noise_0p005) performed just slightly worse than raw data. raised the MAE from 1.1432 to 1.1644 (Degradation: +0.0213)


### For suite experimental:

#### Where data augumentation thrives:
- RandomForest 
The default / mixed pipeline (S9_all_default) dropped the MAE from 1.0646 to 0.9206 (Improvement: -0.1439).

#### Where data augmentation hurt:
- DecisionTree
The maximum noise strategy (S6_interp_noise_0p1) disrupted the model. raised the MAE from 0.0625 to 0.2453 (Degradation: +0.1828)

- XGBoost
The potential drift only strategy (S8_interp_drift_only), caused the model to lose accuracy without the balancing effects of noise or baseline variation. raised the MAE from 1.8375 to 1.9553 (Degradation: +0.1178)

### 11.3 Visualisation - Protocol C MAE by strategy and model

Two isolated Plotly figures. The first is a **grouped bar chart** of Protocol C
MAE across every (strategy, model) combination - look for **shorter bars than
the S0 baseline**. The second is a **line chart of the noise-level sweep**
(S2-S6) so the noise-sensitivity curve is unambiguous.


In [44]:
# Grouped bar chart: Protocol C MAE per (strategy, model)
ablation_mae_figure = go.Figure()
model_color_palette = {
    'Ridge':         '#1f77b4',
    'DecisionTree':  '#ff7f0e',
    'RandomForest':  '#2ca02c',
    'XGBoost':       '#d62728',
}

for feature_suite in FEATURE_SUITES_TO_EVALUATE:
    # Filter the dataframe for the current suite
    suite_df = ablation_results_dataframe[ablation_results_dataframe['Feature Suite'] == feature_suite]
    
    ablation_mae_figure = go.Figure()

    for model_name in ABLATION_MODELS.keys():
        rows_for_this_model = suite_df[
            suite_df['Model'] == model_name
        ]
        ablation_mae_figure.add_trace(go.Bar(
            x=rows_for_this_model['Strategy'],
            y=rows_for_this_model['Protocol C MAE'],
            name=model_name,
            marker_color=model_color_palette[model_name],
            opacity=0.85,
        ))

    ablation_mae_figure.update_layout(
        barmode='group',
        height=PLOTLY_FIGURE_HEIGHT_PX,
        template=PLOTLY_TEMPLATE,
        title=dict(text=f'Figure 11.3.a — Protocol C MAE by strategy and model [{feature_suite.upper()}]',
                   x=0.5, xanchor='center'),
        xaxis_title='Augmentation strategy',
        yaxis_title='Protocol C MAE (µM)',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1.0),
        margin=dict(l=60, r=30, t=70, b=120),
    )
    ablation_mae_figure.update_xaxes(tickangle=-35, showgrid=False)
    ablation_mae_figure.update_yaxes(showgrid=True, minor=dict(showgrid=True))
    ablation_mae_figure.show()


### 11.4 ΔMAE heatmap - quick read of which combinations win/lose

A single heatmap of `ΔMAE (C - A)` with strategies on the y-axis and models on
the x-axis. 

* **Blue cells (Negative ΔMAE)** = Error decreased. The data augmentation **helped**.
* **Red cells (Positive ΔMAE)** = Error increased. The data augmentation **hurt**.


In [47]:
# Loop through each suite to generate a separate heatmap
for feature_suite in FEATURE_SUITES_TO_EVALUATE:
    # Filter the dataframe for the current suite
    suite_df = ablation_results_dataframe[ablation_results_dataframe['Feature Suite'] == feature_suite]
    
    # Build the pivot table using the filtered suite data
    delta_MAE_pivot_table = suite_df.pivot_table(
        index='Strategy', columns='Model', values='ΔMAE (C - A)', aggfunc='first',
    ).reindex(index=[strat_cfg[0] for strat_cfg in ABLATION_STRATEGIES_CONFIG])

    # Build the heatmap - reversed colour scale so negative = Blue (Good), positive = Red (Bad)
    max_absolute_delta_MAE = max(
        abs(delta_MAE_pivot_table.values.min()), abs(delta_MAE_pivot_table.values.max())
    )

    delta_MAE_heatmap_figure = go.Figure(data=go.Heatmap(
        z=delta_MAE_pivot_table.values,
        x=delta_MAE_pivot_table.columns,
        y=delta_MAE_pivot_table.index,
        colorscale='RdBu_r', # REVERSED COLOR TO HAVE NEGATIVE = BLUE (GOOD) AND POSITIVE = RED (BAD)
        zmid=0.0,
        zmin=-max_absolute_delta_MAE, zmax=+max_absolute_delta_MAE,
        text=np.round(delta_MAE_pivot_table.values, 4),
        texttemplate='%{text:+.3f}',
        hovertemplate='Strategy: %{y}<br>Model: %{x}<br>ΔMAE = %{z:+.4f}<extra></extra>',
        colorbar=dict(title='<b>ΔMAE</b><br>Red = Hurt<br>Blue = Helped'),
    ))
    
    delta_MAE_heatmap_figure.update_layout(
        height=PLOTLY_FIGURE_HEIGHT_PX + 60,
        template=PLOTLY_TEMPLATE,
        title=dict(text=f'Figure 11.4 — ΔMAE heatmap (negative = improvement) [{feature_suite.upper()}]',
                   x=0.5, xanchor='center'),
        margin=dict(l=180, r=30, t=70, b=50),
    )
    delta_MAE_heatmap_figure.show()

### 11.5 Verdict - one-line summary per ingredient

For each ingredient we average the ΔMAE across the four models, comparing the
*isolated* strategy against the no-augmentation baseline. This tells us, in one
number, whether the ingredient is **net-helpful**, **net-neutral**, or
**net-harmful**.


In [48]:
# Loop through each suite to generate a separate verdict table
for feature_suite in FEATURE_SUITES_TO_EVALUATE:
    print('=' * 90)
    print(f' FEATURE SUITE: {feature_suite.upper()}')
    print('=' * 90)
    
    # 1. Filter the dataframe for the current suite
    suite_df = ablation_results_dataframe[ablation_results_dataframe['Feature Suite'] == feature_suite]
    
    # 2. Define the helper function locally so it only averages the current suite's data
    def average_delta_MAE_for_strategy(strategy_id: str) -> float:
        return suite_df[
            suite_df['Strategy'] == strategy_id
        ]['ΔMAE (C - A)'].mean()

    # 3. Build the rows using the suite-specific function
    per_ingredient_verdict_rows = [
        {
            'Ingredient':           'Interpolation only',
            'Strategy ID':          'S1_interp_only',
            'Mean ΔMAE (4 models)': round(average_delta_MAE_for_strategy('S1_interp_only'), 4),
        },
        {
            'Ingredient':           'Gaussian noise σ = 0.001 µA',
            'Strategy ID':          'S2_interp_noise_0p001',
            'Mean ΔMAE (4 models)': round(average_delta_MAE_for_strategy('S2_interp_noise_0p001'), 4),
        },
        {
            'Ingredient':           'Gaussian noise σ = 0.01 µA',
            'Strategy ID':          'S4_interp_noise_0p01',
            'Mean ΔMAE (4 models)': round(average_delta_MAE_for_strategy('S4_interp_noise_0p01'), 4),
        },
        {
            'Ingredient':           'Gaussian noise σ = 0.05 µA',
            'Strategy ID':          'S5_interp_noise_0p05',
            'Mean ΔMAE (4 models)': round(average_delta_MAE_for_strategy('S5_interp_noise_0p05'), 4),
        },
        {
            'Ingredient':           'Gaussian noise σ = 0.1 µA',
            'Strategy ID':          'S6_interp_noise_0p1',
            'Mean ΔMAE (4 models)': round(average_delta_MAE_for_strategy('S6_interp_noise_0p1'), 4),
        },
        {
            'Ingredient':           'Baseline variation only (0.05 µA)',
            'Strategy ID':          'S7_interp_baseline_only',
            'Mean ΔMAE (4 models)': round(average_delta_MAE_for_strategy('S7_interp_baseline_only'), 4),
        },
        {
            'Ingredient':           'Potential drift only',
            'Strategy ID':          'S8_interp_drift_only',
            'Mean ΔMAE (4 models)': round(average_delta_MAE_for_strategy('S8_interp_drift_only'), 4),
        },
        {
            'Ingredient':           'ALL ingredients (default pipeline)',
            'Strategy ID':          'S9_all_default',
            'Mean ΔMAE (4 models)': round(average_delta_MAE_for_strategy('S9_all_default'), 4),
        },
    ]

    # 4. Generate and display the final verdict dataframe
    per_ingredient_verdict_dataframe = pd.DataFrame(per_ingredient_verdict_rows)
    per_ingredient_verdict_dataframe['Verdict'] = per_ingredient_verdict_dataframe[
        'Mean ΔMAE (4 models)'
    ].apply(lambda dx: 'HELPFUL ✔' if dx < -1e-3 else ('HARMFUL ✘' if dx > 1e-3 else 'NEUTRAL ~'))

    print(f'Per-ingredient ablation verdict [{feature_suite.upper()}] '
          '(averaged over Ridge / DecisionTree / RandomForest / XGBoost):')
    
    from IPython.display import display # Ensure display is imported for Jupyter
    display(per_ingredient_verdict_dataframe)
    print("\n") # Add a little breathing room between tables

 FEATURE SUITE: CORE
Per-ingredient ablation verdict [CORE] (averaged over Ridge / DecisionTree / RandomForest / XGBoost):


,Ingredient,Strategy ID,Mean ΔMAE (4 models),Verdict
0,Interpolation only,S1_interp_only,-0.0504,HELPFUL ✔
1,Gaussian noise σ = 0.001 µA,S2_interp_noise_0p001,0.0991,HARMFUL ✘
2,Gaussian noise σ = 0.01 µA,S4_interp_noise_0p01,0.0910,HARMFUL ✘
3,Gaussian noise σ = 0.05 µA,S5_interp_noise_0p05,0.0814,HARMFUL ✘
4,Gaussian noise σ = 0.1 µA,S6_interp_noise_0p1,-0.0785,HELPFUL ✔
5,Baseline variation only (0.05 µA),S7_interp_baseline_only,-0.0761,HELPFUL ✔
6,Potential drift only,S8_interp_drift_only,-0.0959,HELPFUL ✔
7,ALL ingredients (default pipeline),S9_all_default,-0.1157,HELPFUL ✔




 FEATURE SUITE: EXTENDED
Per-ingredient ablation verdict [EXTENDED] (averaged over Ridge / DecisionTree / RandomForest / XGBoost):


,Ingredient,Strategy ID,Mean ΔMAE (4 models),Verdict
0,Interpolation only,S1_interp_only,-0.0849,HELPFUL ✔
1,Gaussian noise σ = 0.001 µA,S2_interp_noise_0p001,0.0530,HARMFUL ✘
2,Gaussian noise σ = 0.01 µA,S4_interp_noise_0p01,0.0800,HARMFUL ✘
3,Gaussian noise σ = 0.05 µA,S5_interp_noise_0p05,-0.1088,HELPFUL ✔
4,Gaussian noise σ = 0.1 µA,S6_interp_noise_0p1,0.2167,HARMFUL ✘
5,Baseline variation only (0.05 µA),S7_interp_baseline_only,-0.1256,HELPFUL ✔
6,Potential drift only,S8_interp_drift_only,-0.0774,HELPFUL ✔
7,ALL ingredients (default pipeline),S9_all_default,-0.0921,HELPFUL ✔




 FEATURE SUITE: EXPERIMENTAL
Per-ingredient ablation verdict [EXPERIMENTAL] (averaged over Ridge / DecisionTree / RandomForest / XGBoost):


,Ingredient,Strategy ID,Mean ΔMAE (4 models),Verdict
0,Interpolation only,S1_interp_only,-0.0734,HELPFUL ✔
1,Gaussian noise σ = 0.001 µA,S2_interp_noise_0p001,-0.0435,HELPFUL ✔
2,Gaussian noise σ = 0.01 µA,S4_interp_noise_0p01,-0.0452,HELPFUL ✔
3,Gaussian noise σ = 0.05 µA,S5_interp_noise_0p05,-0.1183,HELPFUL ✔
4,Gaussian noise σ = 0.1 µA,S6_interp_noise_0p1,-0.0489,HELPFUL ✔
5,Baseline variation only (0.05 µA),S7_interp_baseline_only,-0.0299,HELPFUL ✔
6,Potential drift only,S8_interp_drift_only,0.0418,HARMFUL ✘
7,ALL ingredients (default pipeline),S9_all_default,-0.0099,HELPFUL ✔


### 11.6 Direct comparison - best isolated ingredient vs. full pipeline

The single most important question: **does combining all four ingredients
outperform the best individual ingredient, or are we paying a complexity tax
for nothing?**


In [50]:
# Loop through each suite to generate a separate verdict table and conclusion
for feature_suite in FEATURE_SUITES_TO_EVALUATE:
    print('=' * 90)
    print(f' FEATURE SUITE: {feature_suite.upper()}')
    print('=' * 90)
    
    suite_df = ablation_results_dataframe[ablation_results_dataframe['Feature Suite'] == feature_suite]
    
    def average_delta_MAE_for_strategy(strategy_id: str) -> float:
        return suite_df[suite_df['Strategy'] == strategy_id]['ΔMAE (C - A)'].mean()

    per_ingredient_verdict_rows = [
        {'Ingredient': 'Interpolation only',                 'Strategy ID': 'S1_interp_only',          'Mean ΔMAE (4 models)': round(average_delta_MAE_for_strategy('S1_interp_only'), 4)},
        {'Ingredient': 'Gaussian noise σ = 0.001 µA',        'Strategy ID': 'S2_interp_noise_0p001',   'Mean ΔMAE (4 models)': round(average_delta_MAE_for_strategy('S2_interp_noise_0p001'), 4)},
        {'Ingredient': 'Gaussian noise σ = 0.01 µA',         'Strategy ID': 'S4_interp_noise_0p01',    'Mean ΔMAE (4 models)': round(average_delta_MAE_for_strategy('S4_interp_noise_0p01'), 4)},
        {'Ingredient': 'Gaussian noise σ = 0.05 µA',         'Strategy ID': 'S5_interp_noise_0p05',    'Mean ΔMAE (4 models)': round(average_delta_MAE_for_strategy('S5_interp_noise_0p05'), 4)},
        {'Ingredient': 'Gaussian noise σ = 0.1 µA',          'Strategy ID': 'S6_interp_noise_0p1',     'Mean ΔMAE (4 models)': round(average_delta_MAE_for_strategy('S6_interp_noise_0p1'), 4)},
        {'Ingredient': 'Baseline variation only (0.05 µA)',  'Strategy ID': 'S7_interp_baseline_only', 'Mean ΔMAE (4 models)': round(average_delta_MAE_for_strategy('S7_interp_baseline_only'), 4)},
        {'Ingredient': 'Potential drift only',               'Strategy ID': 'S8_interp_drift_only',    'Mean ΔMAE (4 models)': round(average_delta_MAE_for_strategy('S8_interp_drift_only'), 4)},
        {'Ingredient': 'ALL ingredients (default pipeline)', 'Strategy ID': 'S9_all_default',          'Mean ΔMAE (4 models)': round(average_delta_MAE_for_strategy('S9_all_default'), 4)},
    ]

    per_ingredient_verdict_dataframe = pd.DataFrame(per_ingredient_verdict_rows)
    per_ingredient_verdict_dataframe['Verdict'] = per_ingredient_verdict_dataframe[
        'Mean ΔMAE (4 models)'
    ].apply(lambda dx: 'HELPFUL ✔' if dx < -1e-3 else ('HARMFUL ✘' if dx > 1e-3 else 'NEUTRAL ~'))

    print(f'Per-ingredient ablation verdict [{feature_suite.upper()}]:')
    
    from IPython.display import display
    display(per_ingredient_verdict_dataframe)
    
    best_isolated_strategy_id = (
        per_ingredient_verdict_dataframe.iloc[:-1]   # drop the 'all ingredients' row
        .sort_values('Mean ΔMAE (4 models)')
        .iloc[0]['Strategy ID']
    )
    all_default_mean_delta_MAE  = per_ingredient_verdict_dataframe[
        per_ingredient_verdict_dataframe['Strategy ID'] == 'S9_all_default'
    ]['Mean ΔMAE (4 models)'].iloc[0]
    best_isolated_mean_delta_MAE = per_ingredient_verdict_dataframe[
        per_ingredient_verdict_dataframe['Strategy ID'] == best_isolated_strategy_id
    ]['Mean ΔMAE (4 models)'].iloc[0]

    print(f'\nBest isolated ingredient   : {best_isolated_strategy_id}   '
          f'mean ΔMAE = {best_isolated_mean_delta_MAE:+.4f}')
    print(f'Full default pipeline      : S9_all_default                 '
          f'mean ΔMAE = {all_default_mean_delta_MAE:+.4f}')
    
    print('\nOVERALL CONCLUSION:')
    if all_default_mean_delta_MAE < best_isolated_mean_delta_MAE - 1e-3:
        print(f'Stacking all 4 ingredients OUTPERFORMS the best isolated one for the {feature_suite.upper()} suite.')
    elif all_default_mean_delta_MAE > best_isolated_mean_delta_MAE + 1e-3:
        print(f'Stacking all 4 ingredients UNDERPERFORMS the best isolated one for the {feature_suite.upper()} suite. '
              'Consider dropping the harmful ingredient(s).')
    else:
        print(f'Stacking is roughly neutral for the {feature_suite.upper()} suite - the marginal ingredients neither help nor hurt; '
              'drop them for parsimony.')
    print("\n\n")

 FEATURE SUITE: CORE
Per-ingredient ablation verdict [CORE]:


,Ingredient,Strategy ID,Mean ΔMAE (4 models),Verdict
0,Interpolation only,S1_interp_only,-0.0504,HELPFUL ✔
1,Gaussian noise σ = 0.001 µA,S2_interp_noise_0p001,0.0991,HARMFUL ✘
2,Gaussian noise σ = 0.01 µA,S4_interp_noise_0p01,0.0910,HARMFUL ✘
3,Gaussian noise σ = 0.05 µA,S5_interp_noise_0p05,0.0814,HARMFUL ✘
4,Gaussian noise σ = 0.1 µA,S6_interp_noise_0p1,-0.0785,HELPFUL ✔
5,Baseline variation only (0.05 µA),S7_interp_baseline_only,-0.0761,HELPFUL ✔
6,Potential drift only,S8_interp_drift_only,-0.0959,HELPFUL ✔
7,ALL ingredients (default pipeline),S9_all_default,-0.1157,HELPFUL ✔



Best isolated ingredient   : S8_interp_drift_only   mean ΔMAE = -0.0959
Full default pipeline      : S9_all_default                 mean ΔMAE = -0.1157

OVERALL CONCLUSION:
Stacking all 4 ingredients OUTPERFORMS the best isolated one for the CORE suite.



 FEATURE SUITE: EXTENDED
Per-ingredient ablation verdict [EXTENDED]:


,Ingredient,Strategy ID,Mean ΔMAE (4 models),Verdict
0,Interpolation only,S1_interp_only,-0.0849,HELPFUL ✔
1,Gaussian noise σ = 0.001 µA,S2_interp_noise_0p001,0.0530,HARMFUL ✘
2,Gaussian noise σ = 0.01 µA,S4_interp_noise_0p01,0.0800,HARMFUL ✘
3,Gaussian noise σ = 0.05 µA,S5_interp_noise_0p05,-0.1088,HELPFUL ✔
4,Gaussian noise σ = 0.1 µA,S6_interp_noise_0p1,0.2167,HARMFUL ✘
5,Baseline variation only (0.05 µA),S7_interp_baseline_only,-0.1256,HELPFUL ✔
6,Potential drift only,S8_interp_drift_only,-0.0774,HELPFUL ✔
7,ALL ingredients (default pipeline),S9_all_default,-0.0921,HELPFUL ✔



Best isolated ingredient   : S7_interp_baseline_only   mean ΔMAE = -0.1256
Full default pipeline      : S9_all_default                 mean ΔMAE = -0.0921

OVERALL CONCLUSION:
Stacking all 4 ingredients UNDERPERFORMS the best isolated one for the EXTENDED suite. Consider dropping the harmful ingredient(s).



 FEATURE SUITE: EXPERIMENTAL
Per-ingredient ablation verdict [EXPERIMENTAL]:


,Ingredient,Strategy ID,Mean ΔMAE (4 models),Verdict
0,Interpolation only,S1_interp_only,-0.0734,HELPFUL ✔
1,Gaussian noise σ = 0.001 µA,S2_interp_noise_0p001,-0.0435,HELPFUL ✔
2,Gaussian noise σ = 0.01 µA,S4_interp_noise_0p01,-0.0452,HELPFUL ✔
3,Gaussian noise σ = 0.05 µA,S5_interp_noise_0p05,-0.1183,HELPFUL ✔
4,Gaussian noise σ = 0.1 µA,S6_interp_noise_0p1,-0.0489,HELPFUL ✔
5,Baseline variation only (0.05 µA),S7_interp_baseline_only,-0.0299,HELPFUL ✔
6,Potential drift only,S8_interp_drift_only,0.0418,HARMFUL ✘
7,ALL ingredients (default pipeline),S9_all_default,-0.0099,HELPFUL ✔



Best isolated ingredient   : S5_interp_noise_0p05   mean ΔMAE = -0.1183
Full default pipeline      : S9_all_default                 mean ΔMAE = -0.0099

OVERALL CONCLUSION:
Stacking all 4 ingredients UNDERPERFORMS the best isolated one for the EXPERIMENTAL suite. Consider dropping the harmful ingredient(s).



